# NB09：光譜前處理與相似度比對（Preprocessing & Spectral Similarity）— 定性鑑別

**食品分析實驗 週次 11｜對應課堂教材：`lab_decks/w11_preprocess_similarity`**

## 學習目標 (Learning Objectives)

完成本筆記本後，你應該能夠：

1. 對 Raman 光譜做**宇宙射線尖峰移除**（cosmic-spike despiking）與 **ALS/airPLS 基線移除**，並說出 ALS 的參數 λ（平滑度）與 p（不對稱性）如何影響結果、λ 太小為什麼會把真實波峰也當基線吃掉；以及為什麼 airPLS 作為 ALS 的變體只需要調 λ 一個參數（權重由殘差自動調整，不需另設 p）。
2. 對 NIR 光譜做 **SNV**、**MSC**、**Savitzky–Golay 平滑與 1st/2nd 導數**，並說出視窗寬度/多項式階數在「解析度」與「抗雜訊」之間的取捨。
3. 實作四種相似度指標：**Pearson r、cosine、SAM（度）、Euclidean（含/不含正規化）**，並能解釋為什麼原始 Euclidean 距離可能造成排名反轉（ranking flip）。
4. 建立一個小型**光譜庫**（library），對未知樣品做搜尋並回報 **top-3** 結果，而不是只看第一名。
5. 用 **NNLS**（非負最小平方）對簡單混合物做定量預覽，回報比例與 R²。
6. 說出為什麼「光譜庫與待測光譜必須套用完全相同的前處理」。

## 如何使用本筆記本

- 直接「全部執行」（執行階段 → 全部執行）即可跑完整份筆記本，使用內建的**模擬光譜**（非真實量測）。
- `load_spectra()` 支援三種資料來源，用法與 NB01 的 `load_data()` 一致：
  1. 本機/雲端硬碟路徑，例如 `load_spectra("data/nb09_demo_spectra.csv")`。
  2. 已發布的 Google 試算表 CSV 連結。
  3. 若都沒有提供，`load_spectra(None)` 會自動退回本筆記本內嵌的模擬光譜字串。
- 若要換成你們班用 SpectraView 匯出的實際光譜，可用 SpectraView 的「Export combined CSV → rows」功能匯出，`load_spectra`/`split_spectra` 都支援這個格式（細節見下方「光譜資料格式」）。

> ⚠️ **本筆記本內建的資料是模擬光譜（模擬光譜，非真實量測）**，波峰位置/形狀為教學示意，不是文獻定量值。做作業時請換成你們班的實際量測光譜。

### 光譜資料格式 (Shared Spectral Data Format) — NB09–NB13 共用

從本週開始，資料改用**寬格式 (wide format)**，跟 NB01 的長格式不同：**一列＝一條完整光譜**。

| 規則 | 說明 |
|---|---|
| 中繼資料欄位在前 | 欄名**不是數字**的欄位，例如 `sample_id`（樣品代號）、`group`（組別 1–5，沿用 NB01 的定義；TA 提供的參考/示範資料用 `0`）、`instrument`（儀器代號）、`rep`（重複次數）、`role`（`library`/`unknown`/`mixture` 等角色）、以及依主題而定的標籤欄位，例如 `class`（化合物名稱）、`oil`（油品代號）、`acidity_pct`、`aspartame_pct`、`sucralose_pct` 等。 |
| 光譜欄位在後 | 欄名**能轉成浮點數**的欄位＝光譜資料本身，欄名即該點的波長 (nm，NIR) 或波數 (cm⁻¹，Raman)。程式一律依欄名數值排序，不假設檔案裡欄位順序已經由小到大。 |
| 光譜欄位不可缺值 | 每一列在所有光譜欄位都必須有數值，不可有 NaN。 |
| **一個檔案只用一種共同 x 軸** | Raman 位移與 NIR 波長不可混在同一個檔案；不同解析度/範圍的儀器資料也請分開存檔，或先內插到同一組 x 軸再合併。 |
| 標籤欄位的值用英文代碼 | 例如油品用 `EVOO`/`COC`，化合物用 `aspartame`/`sucralose`。畫圖的圖例文字直接取自欄位值，中文會因 Colab 字型問題變成方框。 |

**也相容 SpectraView 的「rows」匯出格式**（`specview/formats/__init__.py` 的 `save_combined_csv(..., layout="rows")`）：
第一列表頭的第一格是 `x:<軸標籤>`（例如 `x:Raman shift (cm-1)`），之後每一格才是 x 值本身；
從第二列開始，每一列的第一格是**光譜名稱**、其餘是強度值。這種格式沒有額外中繼資料欄位，
只有一個 `sample_id`——如果要加上組別/儀器等資訊，請自行用 `sample_id` 做 `merge`。

共用讀取/拆解/畫圖函式（簽名固定，NB09–NB13 都一樣）：

```python
load_spectra(source=None)          # -> 寬格式 DataFrame；None 時退回內建模擬光譜
split_spectra(df)                  # -> (meta, X, axis)
plot_spectra(axis, X, labels)      # -> matplotlib Axes，依 labels 分色、圖例自動去重
```

## 1. 環境設定 (Setup)

只使用 numpy / pandas / scipy / matplotlib / scikit-learn，不需要額外安裝套件。

**注意**：所有圖表座標軸與圖例一律使用**英文**，避免 Colab 預設字型無法顯示中文而出現方框亂碼。

In [ ]:
import io
import os

import numpy as np
import pandas as pd
from scipy import sparse
from scipy.sparse.linalg import spsolve
from scipy.ndimage import median_filter
from scipy.signal import savgol_filter
from scipy.optimize import nnls
import matplotlib.pyplot as plt

plt.style.use("ggplot")
plt.rcParams["axes.unicode_minus"] = False
pd.set_option("display.precision", 4)

RNG_SEED = 42
np.random.seed(RNG_SEED)

In [ ]:
# ---- Embedded demo data (fallback) -------------------------------------
# Exact content of data/nb09_demo_spectra.csv, embedded so the notebook is
# self-contained. 5 個化合物的光譜庫 x 3 重複、3 個未知樣品、2 個混合物。
SAMPLE_CSV = """sample_id,group,instrument,rep,role,class,sucralose_pct,maltodextrin_pct,400,404,408,412,416,420,424,428,432,436,440,444,448,452,456,460,464,468,472,476,480,484,488,492,496,500,504,508,512,516,520,524,528,532,536,540,544,548,552,556,560,564,568,572,576,580,584,588,592,596,600,604,608,612,616,620,624,628,632,636,640,644,648,652,656,660,664,668,672,676,680,684,688,692,696,700,704,708,712,716,720,724,728,732,736,740,744,748,752,756,760,764,768,772,776,780,784,788,792,796,800,804,808,812,816,820,824,828,832,836,840,844,848,852,856,860,864,868,872,876,880,884,888,892,896,900,904,908,912,916,920,924,928,932,936,940,944,948,952,956,960,964,968,972,976,980,984,988,992,996,1000,1004,1008,1012,1016,1020,1024,1028,1032,1036,1040,1044,1048,1052,1056,1060,1064,1068,1072,1076,1080,1084,1088,1092,1096,1100,1104,1108,1112,1116,1120,1124,1128,1132,1136,1140,1144,1148,1152,1156,1160,1164,1168,1172,1176,1180,1184,1188,1192,1196,1200,1204,1208,1212,1216,1220,1224,1228,1232,1236,1240,1244,1248,1252,1256,1260,1264,1268,1272,1276,1280,1284,1288,1292,1296,1300,1304,1308,1312,1316,1320,1324,1328,1332,1336,1340,1344,1348,1352,1356,1360,1364,1368,1372,1376,1380,1384,1388,1392,1396,1400,1404,1408,1412,1416,1420,1424,1428,1432,1436,1440,1444,1448,1452,1456,1460,1464,1468,1472,1476,1480,1484,1488,1492,1496,1500,1504,1508,1512,1516,1520,1524,1528,1532,1536,1540,1544,1548,1552,1556,1560,1564,1568,1572,1576,1580,1584,1588,1592,1596,1600,1604,1608,1612,1616,1620,1624,1628,1632,1636,1640,1644,1648,1652,1656,1660,1664,1668,1672,1676,1680,1684,1688,1692,1696,1700,1704,1708,1712,1716,1720,1724,1728,1732,1736,1740,1744,1748,1752,1756,1760,1764,1768,1772,1776,1780,1784,1788,1792,1796,1800
lib_aspartame_r1,0,Raman785,1,library,aspartame,,,0.14182,0.14431,0.11599,0.12306,0.13796,0.1341,0.13769,0.12991,0.14782,0.14739,0.14086,0.15206,0.14604,0.13335,0.14622,0.13352,0.15247,0.14376,0.14299,0.1386,0.15821,0.14501,0.14284,0.14416,0.15357,0.15247,0.1535,0.15424,0.17191,0.14698,0.14648,0.14402,0.15886,0.16454,0.15266,0.14594,0.14663,0.16192,0.16339,0.16193,0.15041,0.16001,0.15961,0.16168,0.16988,0.1662,0.17545,0.17693,0.19064,0.21019,0.21001,0.24543,0.26871,0.28897,0.30792,0.33141,0.30298,0.30698,0.25949,0.24915,0.23106,0.21555,0.20164,0.19193,0.17388,0.16901,0.18038,0.16918,0.15824,0.15987,0.16235,0.17691,0.17377,0.17966,0.1689,0.17517,0.18024,0.1713,0.17935,0.16856,0.17194,0.17213,0.16437,0.18157,0.17238,0.17756,0.18261,0.18262,0.18516,0.17787,0.17496,0.17873,0.16299,0.16572,0.16728,0.17086,0.18514,0.17239,0.17797,0.19504,0.17877,0.19,0.17357,0.18112,0.17395,0.18032,0.19237,0.16695,0.18882,0.18709,0.17901,0.17072,0.18613,0.18034,0.18817,0.18628,0.20228,0.18407,0.17642,0.18863,0.18922,0.20079,0.19572,0.1911,0.20232,0.17595,0.18159,0.17886,0.18436,0.17462,0.19487,0.18641,0.17403,0.17869,0.19208,0.19742,0.2091,0.21835,0.19343,0.17946,0.1681,0.19216,0.18147,0.18597,0.18689,0.20447,0.26064,0.36579,0.58142,0.86961,1.11644,1.18917,1.02177,0.74494,0.46128,0.30533,0.2169,0.18539,0.18129,0.18243,0.21073,0.18114,0.19767,0.18018,0.19844,0.19289,0.18738,0.18844,0.18219,0.19309,0.18396,0.17614,0.17548,0.18985,0.20378,0.18944,0.1865,0.19039,0.20043,0.18939,0.18291,0.1979,0.19094,0.20182,0.18809,0.17381,0.17217,0.20214,0.20265,0.18339,0.18112,0.19933,0.1734,0.17528,0.1904,0.17977,0.1834,0.18154,0.18628,0.1967,0.18324,0.18848,0.16125,0.18096,0.17271,0.16864,0.17173,0.17685,0.18902,0.16627,0.1795,0.17401,0.17523,0.18818,0.18318,0.19081,0.17553,0.16975,0.17409,0.17838,0.17733,0.16434,0.17569,0.17667,0.19916,0.19235,0.16464,0.16988,0.15771,0.16601,0.17465,0.18312,0.16333,0.16365,0.14827,0.16767,0.15822,0.1631,0.15917,0.16653,0.14943,0.15187,0.18736,0.15272,0.15415,0.183,0.19319,0.15194,0.15948,0.16608,0.17945,0.1518,0.15871,0.16843,0.16449,0.15587,0.15777,0.14484,0.15569,0.15488,0.15934,0.15093,0.16067,0.16554,0.15643,0.15786,0.15433,0.15325,0.14549,0.15532,0.15063,0.17198,0.16623,0.15379,0.14175,0.13769,0.16016,0.15032,0.15193,0.12913,0.15532,0.1501,0.13404,0.14017,0.14753,0.14588,0.14362,0.14781,0.15021,0.16031,0.18231,0.15401,0.19295,0.21618,0.23935,0.25633,0.26611,0.30846,0.33964,0.31799,0.34539,0.32886,0.31939,0.29109,0.27559,0.26713,0.23511,0.22345,0.19762,0.17341,0.1732,0.15016,0.14682,0.1305,0.13066,0.12066,0.13589,0.11304,0.13173,0.12124,0.13419,0.12935,0.13944,0.12794,0.10431,0.11877,0.10713,0.11307,0.13277,0.12345,0.10949,0.10576,0.11563,0.10255,0.10742,0.11666,0.12451,0.11846,0.08888,0.11425,0.11135,0.11419,0.12563,0.08826,0.10241,0.11365,0.09135,0.12136,0.10971,0.11392,0.09918
lib_aspartame_r2,0,Raman785,2,library,aspartame,,,0.11946,0.11997,0.12082,0.10998,0.11763,0.11807,0.12392,0.13058,0.11048,0.12031,0.13686,0.1151,0.1148,0.12553,0.13244,0.12459,0.13825,0.13401,0.13434,0.13194,0.15015,0.1253,0.10779,0.14434,0.12419,0.13032,0.14281,0.11416,0.11813,0.1151,0.12363,0.13643,0.13773,0.13571,0.11928,0.11076,0.13485,0.13004,0.1398,0.14269,0.13753,0.14428,0.13964,0.14363,0.13289,0.1409,0.14937,0.16325,0.16274,0.18825,0.20141,0.22732,0.26199,0.25606,0.27855,0.28251,0.26899,0.27082,0.26362,0.22889,0.21009,0.19878,0.18562,0.16082,0.16821,0.1517,0.14045,0.14208,0.13304,0.13909,0.14466,0.15659,0.16611,0.13542,0.15352,0.13953,0.15502,0.14929,0.13263,0.16055,0.14554,0.14657,0.14153,0.146,0.15713,0.15126,0.15674,0.15737,0.16725,0.15091,0.13985,0.16557,0.15186,0.1666,0.15955,0.15467,0.15973,0.17601,0.17752,0.15769,0.15884,0.16824,0.14926,0.16128,0.15791,0.16153,0.15027,0.14292,0.1383,0.14806,0.15484,0.15669,0.17918,0.17106,0.15055,0.16383,0.15645,0.15784,0.1627,0.1672,0.15776,0.17194,0.15002,0.16164,0.18769,0.16407,0.17629,0.163,0.168,0.16173,0.1607,0.15471,0.1669,0.15417,0.16942,0.1737,0.16658,0.16012,0.16759,0.16002,0.17251,0.1449,0.15996,0.14394,0.15184,0.19354,0.23385,0.33423,0.54958,0.8379,1.12042,1.21213,1.0171,0.70617,0.44376,0.25572,0.19328,0.1721,0.16383,0.17132,0.16663,0.16979,0.15616,0.17196,0.1792,0.15314,0.15389,0.17604,0.16068,0.17654,0.15798,0.17685,0.16351,0.16466,0.17761,0.15822,0.1523,0.15709,0.16597,0.14564,0.16753,0.15562,0.17233,0.13674,0.15266,0.14349,0.15191,0.16247,0.15802,0.15709,0.15784,0.16126,0.14894,0.16589,0.16523,0.16224,0.16374,0.16091,0.17807,0.15661,0.15417,0.14946,0.14643,0.14405,0.14736,0.15528,0.15906,0.15596,0.14752,0.1639,0.16201,0.15274,0.14751,0.15924,0.15533,0.13867,0.15217,0.15516,0.14323,0.15381,0.13704,0.16463,0.16342,0.14808,0.1539,0.12583,0.13802,0.1463,0.13817,0.15568,0.16816,0.13606,0.13909,0.14945,0.16284,0.13413,0.14847,0.16381,0.1287,0.13202,0.1402,0.13884,0.15178,0.14567,0.12511,0.14761,0.13627,0.15438,0.13495,0.13445,0.14581,0.14761,0.14404,0.12227,0.14408,0.12793,0.1402,0.12317,0.14144,0.12847,0.12327,0.14279,0.13762,0.12862,0.14882,0.12945,0.13372,0.13564,0.12629,0.13674,0.12624,0.12699,0.14427,0.11977,0.10558,0.14535,0.15427,0.12425,0.10847,0.12428,0.12408,0.12465,0.1151,0.13187,0.13144,0.11182,0.13508,0.15106,0.13633,0.13752,0.14858,0.16858,0.16118,0.19973,0.2167,0.26333,0.26732,0.30778,0.29771,0.32592,0.32683,0.31036,0.30854,0.30026,0.26186,0.22298,0.2145,0.18214,0.16715,0.15271,0.12305,0.11387,0.12758,0.11247,0.08862,0.11981,0.11313,0.10218,0.09533,0.11619,0.11075,0.13054,0.11041,0.10958,0.10353,0.11287,0.11046,0.11623,0.09801,0.09837,0.09744,0.10965,0.11624,0.09617,0.09602,0.10292,0.09684,0.10833,0.0758,0.08957,0.08952,0.07366,0.08674,0.08675,0.09341,0.10607,0.09201,0.08367,0.09293,0.10352,0.08279
lib_aspartame_r3,0,Raman785,3,library,aspartame,,,0.10976,0.11892,0.11277,0.12039,0.10348,0.11418,0.11265,0.10306,0.10005,0.123,0.11309,0.10684,0.11656,0.12926,0.09562,0.10392,0.11011,0.1344,0.12307,0.12837,0.10974,0.1104,0.12652,0.12307,0.12842,0.1215,0.12659,0.12583,0.13247,0.1332,0.10936,0.10352,0.13645,0.13874,0.11708,0.10911,0.12913,0.13787,0.14696,0.13458,0.12615,0.12144,0.13111,0.12893,0.12327,0.15649,0.15824,0.15889,0.14918,0.18221,0.19968,0.22051,0.2521,0.26676,0.28231,0.28671,0.27908,0.2441,0.24272,0.21357,0.18429,0.19504,0.181,0.1672,0.14978,0.1571,0.1419,0.14306,0.12993,0.14496,0.14185,0.12852,0.14135,0.14139,0.16049,0.15178,0.14327,0.14596,0.14422,0.14206,0.13357,0.14319,0.13753,0.13279,0.15069,0.14978,0.12828,0.1452,0.15666,0.15757,0.15751,0.1479,0.13932,0.13719,0.15173,0.15232,0.16165,0.16002,0.14793,0.13704,0.14652,0.15211,0.14814,0.1446,0.15312,0.14576,0.14472,0.15431,0.14737,0.15403,0.1545,0.14056,0.14732,0.16668,0.14085,0.13147,0.13418,0.15325,0.15342,0.15208,0.16554,0.12681,0.15763,0.16941,0.14264,0.15024,0.14663,0.14532,0.15111,0.16875,0.17294,0.1513,0.1738,0.15519,0.17245,0.15405,0.15636,0.15877,0.18696,0.16374,0.14821,0.16502,0.15181,0.15103,0.16776,0.17122,0.21568,0.32178,0.53328,0.81687,1.0343,1.11826,0.93831,0.65283,0.41222,0.25707,0.17923,0.17101,0.14329,0.15145,0.14945,0.15477,0.15796,0.14504,0.1623,0.15561,0.13599,0.13525,0.15463,0.15245,0.15354,0.15419,0.15663,0.16374,0.14305,0.14232,0.14299,0.15669,0.16612,0.14922,0.12838,0.13621,0.14478,0.14738,0.14872,0.15303,0.14936,0.1628,0.14537,0.14333,0.15656,0.13623,0.15529,0.15222,0.14953,0.16668,0.14437,0.17098,0.14791,0.13717,0.15041,0.13872,0.14174,0.15299,0.15433,0.14231,0.15816,0.1596,0.16214,0.15281,0.16088,0.12817,0.14353,0.13776,0.14734,0.14015,0.14392,0.16411,0.1433,0.14884,0.14207,0.14485,0.14384,0.14795,0.15481,0.15931,0.14561,0.14808,0.13035,0.14065,0.1506,0.14951,0.14263,0.14903,0.14473,0.15147,0.14201,0.1352,0.14176,0.10873,0.1415,0.10082,0.1197,0.14107,0.14096,0.12418,0.1283,0.14874,0.12981,0.15669,0.13385,0.13755,0.14924,0.13399,0.12225,0.13077,0.13111,0.11741,0.12809,0.12281,0.13906,0.12975,0.12615,0.1275,0.13037,0.13388,0.11729,0.11644,0.13747,0.12187,0.11139,0.12957,0.12932,0.10895,0.12611,0.13073,0.12667,0.12881,0.10801,0.12493,0.11818,0.11601,0.12925,0.13518,0.13844,0.13418,0.12116,0.12806,0.136,0.13541,0.14399,0.16252,0.16867,0.18026,0.20442,0.23752,0.2538,0.27774,0.2975,0.29759,0.31225,0.3042,0.27659,0.28597,0.24504,0.20998,0.19253,0.17082,0.16234,0.14317,0.13001,0.13017,0.12058,0.11562,0.11335,0.11326,0.08456,0.10097,0.08205,0.10359,0.10264,0.11264,0.11357,0.10325,0.09575,0.10514,0.09454,0.09934,0.10875,0.09285,0.10602,0.10427,0.08749,0.10631,0.1031,0.09667,0.08757,0.08614,0.0889,0.09925,0.08381,0.09728,0.08004,0.08996,0.09264,0.11555,0.07687,0.10492,0.09125,0.09341,0.09004
lib_sucralose_r1,0,Raman785,1,library,sucralose,,,0.14414,0.1256,0.14205,0.12485,0.13126,0.12171,0.13754,0.14801,0.13335,0.13272,0.12412,0.13354,0.12048,0.12786,0.13985,0.15137,0.16688,0.13354,0.1544,0.14315,0.1399,0.14646,0.14353,0.14484,0.14055,0.13162,0.13124,0.15439,0.14641,0.14469,0.15241,0.16216,0.15815,0.15222,0.17382,0.1805,0.17324,0.1821,0.18939,0.19349,0.19421,0.20065,0.20725,0.19931,0.22643,0.21596,0.23268,0.26581,0.27441,0.29521,0.29306,0.30078,0.32751,0.34958,0.36896,0.40001,0.40636,0.43546,0.44454,0.48179,0.50037,0.51589,0.54386,0.56678,0.58767,0.62895,0.63001,0.64698,0.64532,0.67248,0.69199,0.69887,0.72842,0.71948,0.73812,0.72749,0.72639,0.7218,0.71698,0.72603,0.67532,0.66608,0.66631,0.65579,0.64489,0.62374,0.59508,0.58813,0.56661,0.53609,0.5216,0.49242,0.46198,0.45285,0.42578,0.40415,0.39913,0.34103,0.33348,0.31894,0.3114,0.2978,0.27169,0.26443,0.25427,0.27709,0.26118,0.23064,0.22363,0.20912,0.19566,0.20842,0.19512,0.20036,0.18186,0.18737,0.19404,0.19415,0.19294,0.17412,0.19468,0.17716,0.19071,0.18364,0.16949,0.1797,0.18622,0.18848,0.18078,0.1974,0.19209,0.17942,0.18947,0.20132,0.20161,0.16976,0.17281,0.19697,0.17158,0.16898,0.17739,0.1865,0.18131,0.17587,0.17566,0.17531,0.17366,0.20529,0.18547,0.1914,0.17771,0.18092,0.1761,0.15778,0.1728,0.17304,0.17703,0.20177,0.24903,0.26671,0.32064,0.35912,0.41163,0.43786,0.48184,0.49447,0.47066,0.45212,0.40894,0.34883,0.29564,0.25428,0.23544,0.21562,0.19006,0.18969,0.1904,0.18841,0.19652,0.19314,0.17001,0.19662,0.17427,0.18984,0.17987,0.17488,0.16072,0.17688,0.16279,0.18784,0.19311,0.16975,0.18051,0.17,0.17072,0.1845,0.17612,0.19365,0.17116,0.18571,0.17642,0.18312,0.17041,0.1741,0.17315,0.16565,0.18032,0.1799,0.17902,0.18814,0.18694,0.16837,0.17552,0.16522,0.17135,0.16477,0.18662,0.17371,0.16753,0.18129,0.14728,0.14847,0.1762,0.16846,0.16998,0.15382,0.16646,0.18634,0.17414,0.1672,0.1607,0.16073,0.14297,0.16642,0.17188,0.14724,0.15595,0.15056,0.15204,0.16179,0.13997,0.16651,0.16745,0.15695,0.13419,0.14858,0.17041,0.12983,0.15366,0.14487,0.16424,0.14896,0.15927,0.1604,0.17262,0.15173,0.15703,0.13792,0.16418,0.14417,0.14566,0.15054,0.13463,0.15203,0.15891,0.15164,0.13231,0.14644,0.15457,0.12785,0.13535,0.15434,0.14774,0.14863,0.16257,0.14813,0.14948,0.16947,0.13968,0.1333,0.14501,0.14458,0.13437,0.12877,0.16164,0.14126,0.13012,0.14604,0.13524,0.13324,0.14084,0.12174,0.13762,0.15267,0.13764,0.12865,0.12678,0.12718,0.12818,0.12638,0.13876,0.1117,0.13708,0.1241,0.1311,0.11549,0.13948,0.13808,0.13465,0.10934,0.13221,0.1274,0.10673,0.12197,0.12528,0.12663,0.12227,0.12284,0.1341,0.13106,0.08956,0.11449,0.11545,0.099,0.1169,0.12788,0.10396,0.11763,0.10454,0.10641,0.12734,0.10658,0.11607,0.09217,0.12895,0.11579,0.10738,0.1144,0.09399,0.11936,0.10374,0.10439,0.10096,0.09952,0.11172,0.10679,0.1102,0.10305,0.10772,0.09786,0.1238
lib_sucralose_r2,0,Raman785,2,library,sucralose,,,0.23126,0.21012,0.242,0.23073,0.23155,0.22318,0.22179,0.22293,0.22999,0.24116,0.23002,0.22016,0.23858,0.24062,0.23903,0.24102,0.23195,0.22647,0.22819,0.22166,0.24155,0.23954,0.23721,0.21334,0.23176,0.25686,0.2405,0.25236,0.23657,0.26674,0.25295,0.25954,0.28237,0.27702,0.26185,0.26238,0.26886,0.28145,0.28004,0.28243,0.30696,0.29771,0.32913,0.3151,0.34094,0.33644,0.35319,0.34798,0.35934,0.38613,0.40609,0.4123,0.4157,0.43258,0.45042,0.46635,0.50517,0.536,0.54576,0.54478,0.58713,0.60906,0.6531,0.63862,0.66974,0.71033,0.71638,0.73488,0.74634,0.76283,0.77986,0.7746,0.80004,0.78392,0.8051,0.80549,0.79712,0.80036,0.78943,0.77785,0.77587,0.75728,0.75868,0.72947,0.73218,0.6961,0.68408,0.66451,0.6644,0.63446,0.60884,0.59573,0.54963,0.55057,0.55175,0.51448,0.49594,0.48366,0.47094,0.42676,0.41565,0.42132,0.39982,0.40885,0.38232,0.35683,0.35511,0.3328,0.35127,0.34583,0.33828,0.34878,0.34605,0.32898,0.30645,0.33152,0.32885,0.32467,0.32998,0.30736,0.33817,0.30776,0.32428,0.29948,0.30674,0.33099,0.31308,0.32401,0.31796,0.31456,0.3293,0.31337,0.29802,0.3079,0.32156,0.32012,0.30061,0.30915,0.31069,0.31296,0.30295,0.31052,0.3096,0.32468,0.32512,0.31563,0.31321,0.31857,0.30904,0.30577,0.3079,0.31112,0.30625,0.32046,0.32823,0.30924,0.33305,0.34363,0.36952,0.39482,0.41773,0.46312,0.52091,0.57049,0.56537,0.60063,0.59908,0.55802,0.52514,0.46742,0.42945,0.37679,0.34039,0.35266,0.32003,0.31731,0.3035,0.30085,0.31193,0.30564,0.31623,0.31659,0.31491,0.31374,0.32677,0.29995,0.3097,0.32119,0.32211,0.29484,0.30552,0.32179,0.31286,0.30928,0.3047,0.30257,0.30881,0.30292,0.30452,0.30523,0.29715,0.30873,0.33202,0.28456,0.30502,0.29362,0.30643,0.30054,0.30711,0.31065,0.30021,0.30899,0.27361,0.31232,0.29953,0.30591,0.28227,0.2855,0.29459,0.29228,0.29459,0.30191,0.27271,0.29938,0.28901,0.28616,0.27274,0.29174,0.26182,0.29168,0.26951,0.28419,0.28168,0.29378,0.2795,0.27355,0.27461,0.29159,0.2834,0.27335,0.27401,0.25986,0.27856,0.25623,0.28048,0.27338,0.29509,0.25934,0.26843,0.28655,0.27958,0.2598,0.26809,0.26496,0.26684,0.25611,0.26128,0.24525,0.26165,0.26385,0.25613,0.2413,0.26637,0.25917,0.2473,0.25078,0.2376,0.24138,0.24527,0.26194,0.24227,0.25407,0.24129,0.24815,0.24325,0.25238,0.23729,0.2259,0.22679,0.23426,0.22607,0.24163,0.23184,0.25921,0.21705,0.23543,0.22107,0.22637,0.23167,0.20596,0.23053,0.24614,0.22152,0.2276,0.2342,0.22125,0.21602,0.19801,0.22611,0.21751,0.20745,0.21778,0.20961,0.20909,0.21937,0.19885,0.22164,0.19612,0.2024,0.22718,0.21816,0.20679,0.19533,0.19837,0.20333,0.2019,0.19954,0.19907,0.19616,0.18282,0.191,0.20424,0.18035,0.19107,0.1837,0.19194,0.18588,0.20347,0.1927,0.17346,0.19222,0.17671,0.17763,0.18548,0.17637,0.18916,0.17143,0.17194,0.17052,0.17038,0.18598,0.18024,0.19001,0.19455,0.16053,0.1705,0.15701,0.15996,0.16041,0.15512,0.15415
lib_sucralose_r3,0,Raman785,3,library,sucralose,,,0.08524,0.0679,0.10544,0.0731,0.0786,0.06519,0.09548,0.08988,0.08192,0.06576,0.08161,0.06856,0.09468,0.08844,0.08159,0.05658,0.07142,0.08846,0.09406,0.0934,0.06679,0.08616,0.09378,0.06757,0.08468,0.08817,0.0832,0.06857,0.08485,0.08601,0.08917,0.08659,0.08962,0.09576,0.10303,0.09838,0.10056,0.1093,0.09341,0.12223,0.12294,0.12953,0.14571,0.12436,0.13283,0.16492,0.17271,0.1861,0.19246,0.21747,0.23499,0.24641,0.25593,0.25543,0.30312,0.32744,0.34224,0.34822,0.3855,0.40947,0.43874,0.43668,0.46526,0.49666,0.51078,0.5192,0.55361,0.57303,0.5844,0.60028,0.60819,0.6362,0.62087,0.63018,0.63997,0.65005,0.62526,0.62368,0.6106,0.6232,0.60142,0.59644,0.59269,0.56375,0.55506,0.53104,0.52095,0.49832,0.46535,0.44886,0.43906,0.41063,0.37098,0.37013,0.33186,0.33662,0.3133,0.28739,0.26845,0.24818,0.2415,0.21653,0.20919,0.20323,0.17762,0.16682,0.15178,0.14475,0.12966,0.14863,0.15357,0.14459,0.10958,0.12918,0.13278,0.11109,0.11419,0.09099,0.11195,0.09715,0.10854,0.0899,0.11009,0.11421,0.09557,0.09158,0.08103,0.09964,0.09627,0.08641,0.10633,0.0972,0.09088,0.09605,0.11228,0.10759,0.10025,0.10775,0.08995,0.1019,0.10306,0.0969,0.10597,0.09383,0.10674,0.10717,0.07249,0.11358,0.07121,0.09555,0.11203,0.09879,0.08774,0.09251,0.11596,0.09095,0.12966,0.1317,0.1419,0.17867,0.21399,0.27904,0.32539,0.34298,0.38523,0.39101,0.40212,0.34651,0.33387,0.26972,0.21214,0.17724,0.14524,0.11886,0.11859,0.09708,0.083,0.09868,0.08333,0.0933,0.09148,0.09171,0.12032,0.10078,0.10258,0.08227,0.0934,0.102,0.0895,0.10872,0.08751,0.09657,0.09346,0.10086,0.09269,0.09627,0.0965,0.0914,0.10347,0.10088,0.09748,0.09393,0.10607,0.09561,0.09476,0.09873,0.11451,0.084,0.10495,0.07817,0.09913,0.11255,0.10153,0.09819,0.09072,0.09648,0.09348,0.08798,0.09277,0.09748,0.08765,0.08709,0.08137,0.09871,0.10007,0.09675,0.1039,0.09494,0.0978,0.09673,0.09258,0.088,0.11525,0.08839,0.08605,0.09685,0.09483,0.07806,0.10313,0.09075,0.08038,0.10316,0.09732,0.0782,0.10841,0.09701,0.0901,0.10588,0.09135,0.08926,0.08271,0.09491,0.09363,0.08878,0.0805,0.08947,0.08724,0.07437,0.07276,0.09289,0.06098,0.08882,0.08174,0.09937,0.08111,0.09239,0.08159,0.08645,0.09243,0.08019,0.09454,0.07439,0.08702,0.09305,0.09017,0.10249,0.07543,0.06902,0.07262,0.08891,0.08703,0.08281,0.07031,0.08895,0.06877,0.09235,0.10907,0.0839,0.08581,0.07479,0.06135,0.07317,0.068,0.0642,0.09563,0.08087,0.08167,0.07452,0.07561,0.08672,0.07649,0.08185,0.07432,0.07259,0.06168,0.0557,0.0726,0.07551,0.06182,0.07407,0.07468,0.05232,0.06446,0.07741,0.07694,0.06127,0.07412,0.0825,0.07977,0.07078,0.06679,0.06491,0.07561,0.07262,0.0767,0.07146,0.07806,0.07055,0.07719,0.07778,0.06077,0.05046,0.08372,0.07059,0.05061,0.06035,0.07495,0.06417,0.06159,0.08045,0.05571,0.05794,0.06778,0.06931,0.07011,0.074,0.06662,0.06927,0.06153,0.06449,0.06468
lib_sucrose_r1,0,Raman785,1,library,sucrose,,,0.13776,0.13323,0.12527,0.13462,0.13424,0.13703,0.15774,0.13101,0.14105,0.14809,0.13895,0.16192,0.13534,0.14293,0.13227,0.13731,0.14388,0.15042,0.14571,0.15567,0.14269,0.14087,0.17179,0.14925,0.13867,0.13947,0.13754,0.15688,0.14441,0.14988,0.14851,0.157,0.14176,0.14,0.15354,0.15784,0.15568,0.14479,0.16018,0.14649,0.14923,0.14553,0.16849,0.16233,0.16255,0.16948,0.18564,0.14092,0.15113,0.15284,0.15099,0.14921,0.15748,0.14879,0.16478,0.16243,0.16719,0.16793,0.16935,0.16005,0.18545,0.16704,0.14652,0.15557,0.14864,0.16666,0.1657,0.16267,0.18789,0.1737,0.17589,0.15974,0.15947,0.17024,0.17201,0.15212,0.16928,0.17161,0.16971,0.16411,0.16723,0.17123,0.17428,0.17979,0.18491,0.17152,0.17025,0.17954,0.18162,0.1715,0.18408,0.18631,0.18975,0.19494,0.16881,0.17792,0.17821,0.17744,0.17231,0.16454,0.18155,0.17578,0.19076,0.19534,0.21712,0.27097,0.3275,0.40101,0.48588,0.5183,0.53968,0.50521,0.45207,0.41243,0.31859,0.26049,0.23049,0.2056,0.18458,0.18469,0.19817,0.17886,0.17827,0.21529,0.2392,0.27668,0.33421,0.41432,0.47378,0.49792,0.44197,0.3864,0.32226,0.23316,0.22943,0.1873,0.19375,0.18977,0.19723,0.19171,0.16244,0.17841,0.18769,0.1844,0.19116,0.19431,0.19253,0.18432,0.16802,0.18193,0.19066,0.17559,0.19279,0.19555,0.18771,0.15578,0.17361,0.20521,0.18084,0.16641,0.19855,0.18627,0.19712,0.18876,0.1905,0.18946,0.19272,0.20844,0.19532,0.16986,0.19046,0.19843,0.20157,0.19081,0.21758,0.23454,0.28215,0.29422,0.35564,0.42566,0.49155,0.54281,0.55521,0.5783,0.55484,0.49057,0.44675,0.35915,0.31255,0.25658,0.2307,0.20552,0.20518,0.18615,0.18765,0.18495,0.1777,0.18421,0.20218,0.17684,0.18345,0.19155,0.19008,0.1785,0.15425,0.18295,0.15528,0.19185,0.17853,0.18651,0.16716,0.16032,0.17811,0.17483,0.1699,0.17531,0.20178,0.18209,0.18387,0.18144,0.18742,0.17035,0.15131,0.16959,0.1792,0.16098,0.16852,0.16296,0.18448,0.17655,0.17515,0.17443,0.16741,0.16933,0.15507,0.14462,0.16754,0.16884,0.17724,0.18041,0.17994,0.1538,0.1598,0.16766,0.1877,0.17671,0.17263,0.15708,0.15164,0.16788,0.14849,0.15662,0.15736,0.17394,0.15396,0.13262,0.16155,0.149,0.15591,0.1541,0.15171,0.16183,0.13808,0.14771,0.1592,0.14569,0.13521,0.15745,0.16146,0.14924,0.12954,0.15566,0.13146,0.1506,0.13809,0.13482,0.14144,0.14348,0.13831,0.15008,0.13918,0.15288,0.14533,0.13498,0.13273,0.12435,0.14316,0.13614,0.11539,0.13319,0.12896,0.15316,0.1198,0.12184,0.14678,0.14867,0.12707,0.12762,0.13953,0.10951,0.11359,0.12882,0.14654,0.1261,0.13089,0.14063,0.14978,0.12236,0.12738,0.14518,0.11116,0.13607,0.11503,0.12565,0.12162,0.11388,0.11304,0.12505,0.13166,0.11983,0.12124,0.12567,0.10966,0.12619,0.1264,0.14367,0.11329,0.11189,0.12358,0.1235,0.10938,0.11013,0.11899,0.10473,0.10832,0.11275,0.12124,0.10467,0.09172,0.0904,0.09641,0.0953,0.10632,0.09541,0.09971,0.09843,0.10121,0.11828,0.11111,0.10409,0.09877
lib_sucrose_r2,0,Raman785,2,library,sucrose,,,0.21764,0.20375,0.21422,0.19457,0.21803,0.22456,0.21517,0.23716,0.2112,0.22321,0.2217,0.22207,0.22295,0.22617,0.23459,0.2409,0.22811,0.23729,0.22312,0.23879,0.22451,0.24403,0.25622,0.23388,0.21416,0.23159,0.25037,0.24455,0.24248,0.25105,0.24677,0.25507,0.25304,0.26002,0.24934,0.25575,0.24457,0.24256,0.26658,0.2526,0.25941,0.25352,0.24733,0.25422,0.26937,0.26147,0.23969,0.25555,0.27641,0.25733,0.24234,0.28014,0.2689,0.26223,0.26007,0.27546,0.26005,0.27779,0.27256,0.27041,0.27023,0.26784,0.28085,0.26254,0.25576,0.27891,0.27986,0.27912,0.27916,0.27263,0.28446,0.27688,0.2732,0.27568,0.29973,0.26289,0.28594,0.26485,0.28125,0.27868,0.26284,0.28825,0.28122,0.26468,0.29472,0.30131,0.30249,0.2918,0.28928,0.3027,0.30684,0.30194,0.28177,0.29376,0.29671,0.27477,0.29811,0.3167,0.30098,0.31013,0.29865,0.29908,0.31011,0.3114,0.33433,0.39631,0.45346,0.49748,0.5843,0.62412,0.64738,0.63041,0.5862,0.51067,0.45711,0.39083,0.35512,0.31696,0.33445,0.312,0.31276,0.29071,0.30385,0.33788,0.35618,0.38166,0.47671,0.53089,0.60176,0.5985,0.56847,0.51488,0.42432,0.36641,0.33755,0.3284,0.29842,0.29206,0.31615,0.30176,0.30392,0.28937,0.30123,0.30802,0.31747,0.30051,0.32485,0.30147,0.30151,0.32373,0.31047,0.30283,0.30679,0.32184,0.32294,0.30006,0.30774,0.309,0.29816,0.30737,0.30789,0.31912,0.31967,0.30418,0.30519,0.31382,0.30117,0.31275,0.30585,0.297,0.29736,0.32229,0.30862,0.33187,0.3208,0.34757,0.39881,0.40832,0.45727,0.55523,0.59995,0.66171,0.68623,0.69444,0.6514,0.60474,0.54452,0.46315,0.4082,0.36705,0.32533,0.31601,0.31028,0.29915,0.2974,0.29836,0.2911,0.30394,0.31086,0.29135,0.30034,0.29259,0.29812,0.3096,0.29849,0.27513,0.29685,0.30408,0.28896,0.30056,0.28312,0.27457,0.29023,0.27064,0.28519,0.29983,0.26722,0.27288,0.2747,0.28373,0.28004,0.29701,0.29188,0.28659,0.2824,0.28417,0.2709,0.28737,0.26695,0.28569,0.28845,0.27795,0.27332,0.27832,0.24493,0.27579,0.27085,0.27069,0.27746,0.26884,0.25738,0.26655,0.26781,0.26746,0.26709,0.27503,0.25293,0.26469,0.27497,0.24386,0.26214,0.25361,0.25488,0.24865,0.25511,0.27433,0.24237,0.24241,0.24018,0.26956,0.24674,0.25018,0.24548,0.23913,0.24529,0.27128,0.25967,0.23627,0.25032,0.23457,0.22416,0.25116,0.2516,0.24948,0.23335,0.2354,0.24358,0.23902,0.23018,0.21693,0.23327,0.24192,0.21816,0.23552,0.23144,0.23462,0.23483,0.21816,0.20792,0.21774,0.23095,0.21562,0.21308,0.20078,0.2355,0.2252,0.22361,0.22456,0.21962,0.21038,0.21016,0.20649,0.23607,0.21425,0.20302,0.21127,0.19547,0.21959,0.1969,0.20748,0.21209,0.20424,0.2003,0.19842,0.19671,0.19047,0.1907,0.20895,0.19204,0.18423,0.18588,0.20101,0.19573,0.19354,0.18433,0.1916,0.1784,0.18121,0.19247,0.19049,0.1709,0.17114,0.17975,0.1949,0.15747,0.17838,0.16082,0.16722,0.1689,0.16733,0.16518,0.16958,0.16315,0.15287,0.17429,0.17302,0.17292,0.14809,0.16549,0.16827,0.15021
lib_sucrose_r3,0,Raman785,3,library,sucrose,,,0.105,0.10615,0.07058,0.11248,0.07871,0.09799,0.09821,0.07887,0.09407,0.08838,0.09532,0.0924,0.10237,0.09148,0.08936,0.10231,0.09747,0.0993,0.08564,0.08215,0.08723,0.08701,0.10139,0.1036,0.09259,0.10876,0.10844,0.09749,0.08714,0.11141,0.09535,0.09752,0.09316,0.08796,0.08246,0.10377,0.0899,0.12862,0.08746,0.08604,0.104,0.11448,0.10011,0.10547,0.0816,0.09354,0.08986,0.08192,0.10232,0.07899,0.08847,0.09659,0.09392,0.10646,0.10582,0.11197,0.11081,0.11496,0.10379,0.1071,0.10943,0.10599,0.10085,0.10174,0.09714,0.10978,0.10486,0.09941,0.09966,0.09441,0.11654,0.10777,0.09858,0.11552,0.10206,0.09382,0.11832,0.10295,0.10445,0.10916,0.12039,0.10937,0.09738,0.11502,0.10493,0.12885,0.11016,0.08801,0.11413,0.13019,0.12669,0.12942,0.10109,0.12059,0.10442,0.11406,0.09926,0.10573,0.10567,0.12188,0.12199,0.11944,0.10944,0.13298,0.15316,0.20938,0.26924,0.33658,0.3866,0.41773,0.44651,0.4596,0.39065,0.31928,0.27491,0.21461,0.16239,0.13495,0.12368,0.12139,0.13318,0.12237,0.13058,0.14848,0.15321,0.2206,0.27663,0.32551,0.38147,0.40435,0.38614,0.31735,0.24686,0.20642,0.13661,0.12918,0.12384,0.11029,0.117,0.10051,0.13058,0.10969,0.13685,0.11623,0.12686,0.11606,0.1163,0.11078,0.11782,0.11982,0.10879,0.11538,0.11053,0.13962,0.11571,0.13205,0.11462,0.11915,0.10332,0.11203,0.11858,0.1323,0.11862,0.1302,0.12272,0.13254,0.11563,0.14009,0.11133,0.10862,0.10456,0.14104,0.12849,0.12249,0.14915,0.16278,0.20653,0.21638,0.2878,0.35334,0.40548,0.49071,0.50835,0.51835,0.50339,0.42117,0.35303,0.2907,0.24058,0.17984,0.14802,0.14546,0.12763,0.09765,0.10308,0.12907,0.11749,0.10554,0.11126,0.11292,0.11999,0.09889,0.1039,0.11488,0.11098,0.11753,0.09935,0.11504,0.12221,0.11598,0.11232,0.10978,0.10221,0.10964,0.12277,0.10879,0.10088,0.10465,0.11138,0.10543,0.11381,0.11398,0.11743,0.12256,0.11806,0.09854,0.09863,0.10668,0.1027,0.0956,0.10233,0.10177,0.09956,0.10288,0.11759,0.10755,0.13003,0.10677,0.10192,0.10461,0.10688,0.10842,0.12457,0.0951,0.11188,0.10511,0.08705,0.11021,0.11491,0.09339,0.1016,0.08417,0.08856,0.11135,0.09593,0.08923,0.10233,0.09453,0.09476,0.10658,0.08728,0.08956,0.09986,0.10772,0.11012,0.09079,0.10652,0.08901,0.11482,0.08492,0.10925,0.0891,0.11549,0.10097,0.08459,0.08923,0.10195,0.10436,0.09282,0.09367,0.10049,0.09392,0.07936,0.08385,0.09402,0.10351,0.1072,0.09852,0.09328,0.09758,0.08311,0.08793,0.0809,0.07634,0.09184,0.09004,0.09474,0.09301,0.08965,0.09247,0.08616,0.09039,0.07868,0.08931,0.09198,0.07827,0.08517,0.07809,0.07874,0.07243,0.09601,0.09369,0.08808,0.10211,0.08017,0.11206,0.07914,0.08364,0.07679,0.08417,0.08848,0.07866,0.09725,0.08155,0.08154,0.06942,0.08238,0.07653,0.07147,0.06947,0.10363,0.07396,0.0617,0.06897,0.07512,0.08078,0.08055,0.08145,0.06792,0.07746,0.06498,0.06723,0.06393,0.06091,0.08312,0.07906,0.08163,0.07194,0.07901,0.05332,0.07925
lib_erythritol_r1,0,Raman785,1,library,erythritol,,,0.14629,0.13326,0.12879,0.13728,0.15069,0.15258,0.13263,0.13713,0.14035,0.14464,0.13522,0.13525,0.14188,0.15159,0.13955,0.15082,0.16318,0.1402,0.15178,0.16471,0.15089,0.14995,0.16794,0.15385,0.16857,0.16242,0.15531,0.16414,0.16315,0.17075,0.15565,0.16214,0.16171,0.16996,0.15325,0.17426,0.16933,0.15208,0.17791,0.18874,0.17699,0.18181,0.16896,0.15206,0.15079,0.17234,0.16443,0.16175,0.17082,0.17815,0.158,0.16858,0.1887,0.16338,0.18336,0.17034,0.18117,0.1872,0.18415,0.18634,0.16624,0.18939,0.17022,0.18184,0.18232,0.18507,0.18407,0.17274,0.16815,0.19084,0.18444,0.19454,0.18305,0.1699,0.19406,0.18114,0.1887,0.17924,0.20258,0.1778,0.18361,0.18847,0.17688,0.19151,0.20473,0.1841,0.19334,0.16228,0.18976,0.1874,0.18271,0.20207,0.18835,0.19019,0.18854,0.17349,0.18167,0.20293,0.17255,0.1863,0.19028,0.21202,0.20258,0.18962,0.20852,0.18836,0.20614,0.18275,0.20742,0.19315,0.18799,0.19854,0.209,0.21275,0.2445,0.2678,0.32893,0.3864,0.44025,0.48052,0.50297,0.49082,0.46225,0.39253,0.33402,0.30644,0.24642,0.22523,0.21634,0.19408,0.19767,0.19637,0.20777,0.19959,0.2028,0.21239,0.21024,0.20008,0.19507,0.2132,0.20966,0.18592,0.1989,0.20374,0.20991,0.21126,0.20503,0.20807,0.20288,0.19552,0.19969,0.20266,0.20266,0.1857,0.20458,0.21821,0.23302,0.26527,0.32135,0.36966,0.43472,0.50943,0.55703,0.56619,0.52049,0.43591,0.37756,0.3183,0.26862,0.24017,0.20726,0.20783,0.20722,0.20955,0.19255,0.19683,0.17303,0.19367,0.19446,0.18933,0.20099,0.19858,0.17634,0.20088,0.18923,0.20315,0.18525,0.19814,0.19477,0.20039,0.19835,0.1943,0.18122,0.18155,0.19021,0.19568,0.1894,0.20144,0.1921,0.18494,0.1915,0.19623,0.1969,0.21211,0.18631,0.19194,0.19663,0.19187,0.18696,0.17661,0.18769,0.18383,0.18092,0.19682,0.20525,0.23825,0.26185,0.277,0.31892,0.34051,0.38914,0.42523,0.4442,0.42839,0.43047,0.37679,0.34362,0.32476,0.29804,0.25522,0.22559,0.1865,0.17643,0.1905,0.18116,0.17107,0.18124,0.18075,0.18833,0.17967,0.19005,0.17733,0.17559,0.18687,0.16592,0.20333,0.17842,0.17794,0.17886,0.19184,0.18046,0.15594,0.18548,0.16749,0.17142,0.157,0.1745,0.16216,0.16911,0.1647,0.18623,0.18623,0.16629,0.16589,0.16094,0.17515,0.16021,0.1587,0.17211,0.15635,0.16076,0.14473,0.16698,0.17875,0.16595,0.17275,0.14906,0.15928,0.14687,0.15341,0.1628,0.15548,0.13971,0.14007,0.13879,0.15689,0.17199,0.16435,0.13114,0.16368,0.15821,0.13276,0.14455,0.1579,0.15182,0.13259,0.15115,0.12303,0.14154,0.15451,0.14268,0.13686,0.15011,0.15032,0.15303,0.14803,0.14946,0.14533,0.14002,0.13762,0.13371,0.12331,0.12524,0.13133,0.13592,0.14727,0.1318,0.14161,0.14173,0.12798,0.1356,0.12658,0.14159,0.13409,0.11519,0.12807,0.14229,0.12336,0.12901,0.12936,0.1129,0.12773,0.10805,0.1151,0.13148,0.12366,0.11642,0.1233,0.11761,0.1177,0.12352,0.10905,0.11611,0.11196,0.0986,0.10464,0.1289,0.11645,0.09875,0.09597,0.10964
lib_erythritol_r2,0,Raman785,2,library,erythritol,,,0.18912,0.1804,0.18242,0.16518,0.18874,0.17684,0.18964,0.1737,0.18443,0.187,0.18276,0.18916,0.18327,0.18025,0.20361,0.19886,0.18157,0.19567,0.18336,0.17974,0.20273,0.17615,0.20325,0.19225,0.19345,0.20046,0.20296,0.18967,0.20589,0.20062,0.1877,0.19934,0.21562,0.21934,0.21051,0.19219,0.20976,0.21411,0.21361,0.21495,0.21147,0.20786,0.20323,0.19107,0.21417,0.21336,0.21779,0.19956,0.2201,0.2349,0.21584,0.21514,0.21954,0.23746,0.20759,0.22054,0.21118,0.21438,0.22002,0.23465,0.23984,0.22697,0.21937,0.2487,0.22013,0.21933,0.20792,0.24025,0.23658,0.25169,0.21422,0.24329,0.23,0.23795,0.21399,0.24572,0.23947,0.23178,0.23918,0.22562,0.22802,0.22972,0.21636,0.24403,0.24784,0.22943,0.24043,0.23479,0.24122,0.23214,0.23006,0.24989,0.23919,0.26051,0.2412,0.23192,0.25389,0.24836,0.24974,0.25479,0.23677,0.24242,0.24237,0.23156,0.24221,0.25859,0.24796,0.23251,0.23084,0.26366,0.24729,0.24435,0.25397,0.25409,0.30558,0.32405,0.3935,0.44291,0.51886,0.55368,0.57712,0.5588,0.51179,0.44737,0.3795,0.34963,0.28527,0.2724,0.26085,0.25934,0.24888,0.25564,0.25068,0.23662,0.25265,0.24168,0.23546,0.25429,0.2579,0.25839,0.25888,0.23396,0.23975,0.27316,0.24301,0.25964,0.2467,0.25439,0.24729,0.25926,0.25506,0.24939,0.24393,0.25874,0.24816,0.27414,0.31531,0.33619,0.38127,0.44072,0.49659,0.56806,0.63258,0.61175,0.57397,0.5138,0.43022,0.36087,0.32103,0.27876,0.26087,0.25068,0.25929,0.2499,0.24966,0.25703,0.24938,0.24779,0.25504,0.25678,0.24838,0.25041,0.24152,0.24046,0.25648,0.25296,0.25277,0.25072,0.24806,0.25425,0.25631,0.25574,0.24528,0.25981,0.24787,0.25154,0.24767,0.24694,0.23673,0.25071,0.27175,0.23914,0.24318,0.23759,0.23675,0.25665,0.23406,0.23569,0.22433,0.23269,0.22383,0.249,0.2744,0.2496,0.24805,0.29124,0.29959,0.35161,0.37313,0.40917,0.45052,0.48843,0.494,0.49075,0.4772,0.43681,0.41557,0.35308,0.32806,0.30723,0.27694,0.26119,0.24208,0.24645,0.23917,0.23151,0.23458,0.23165,0.21888,0.2091,0.22606,0.22665,0.23142,0.21049,0.21012,0.2231,0.21711,0.19828,0.2202,0.19723,0.2056,0.22787,0.23189,0.20078,0.20892,0.21193,0.22859,0.20512,0.20455,0.22407,0.20409,0.21432,0.19158,0.20389,0.18825,0.19354,0.21173,0.20207,0.19861,0.19355,0.19806,0.2025,0.20854,0.17168,0.18883,0.19795,0.19458,0.19881,0.19666,0.19014,0.18712,0.18914,0.20228,0.18266,0.18855,0.1902,0.20316,0.18374,0.19552,0.19743,0.18366,0.18137,0.17937,0.17928,0.18956,0.1825,0.18172,0.16729,0.16804,0.19832,0.17428,0.16748,0.1887,0.17638,0.17851,0.17052,0.16788,0.16853,0.15557,0.16768,0.17069,0.1854,0.16692,0.14188,0.16345,0.19095,0.17138,0.15378,0.16696,0.1528,0.14585,0.15507,0.15863,0.13817,0.14457,0.15533,0.15032,0.15752,0.14059,0.15905,0.13925,0.16669,0.13228,0.13769,0.15197,0.16362,0.14931,0.15097,0.1447,0.13344,0.13799,0.12996,0.12849,0.14448,0.12843,0.14205,0.12503,0.14021,0.13156,0.14972,0.1381
lib_erythritol_r3,0,Raman785,3,library,erythritol,,,0.07216,0.06202,0.06793,0.06968,0.06528,0.06062,0.05888,0.04985,0.06292,0.07999,0.05796,0.0605,0.09232,0.06828,0.07808,0.08153,0.08196,0.06034,0.07348,0.05984,0.10071,0.06053,0.06623,0.06653,0.06219,0.09091,0.10183,0.07307,0.06485,0.0559,0.07605,0.07555,0.07242,0.08136,0.08332,0.08281,0.07664,0.069,0.08689,0.08057,0.08954,0.08876,0.06685,0.06694,0.07393,0.07113,0.08945,0.08562,0.0933,0.08647,0.08479,0.0949,0.0878,0.08398,0.0983,0.08484,0.08084,0.07569,0.09129,0.09682,0.08413,0.0979,0.06531,0.08463,0.08497,0.07828,0.10957,0.08524,0.082,0.08651,0.0664,0.079,0.0966,0.08064,0.07881,0.10035,0.10082,0.09275,0.08319,0.0855,0.08001,0.07826,0.07791,0.08232,0.0783,0.07316,0.09113,0.09143,0.10623,0.08268,0.09246,0.09276,0.08722,0.09734,0.09777,0.09342,0.08772,0.08682,0.07684,0.08257,0.08453,0.07962,0.09537,0.10305,0.08211,0.09413,0.08107,0.08848,0.0848,0.0981,0.08622,0.1055,0.09851,0.10626,0.12222,0.17327,0.2307,0.28608,0.36695,0.40466,0.40437,0.38732,0.34955,0.27539,0.23398,0.17506,0.13817,0.12062,0.11933,0.09837,0.0933,0.10873,0.10309,0.09016,0.09344,0.09538,0.09708,0.07992,0.0781,0.08467,0.08393,0.08716,0.09996,0.09195,0.09561,0.08142,0.08906,0.09956,0.09964,0.08646,0.09537,0.11592,0.09432,0.08919,0.09857,0.11072,0.12727,0.16174,0.19843,0.26995,0.35163,0.41604,0.45009,0.45395,0.41821,0.36187,0.27837,0.21658,0.15631,0.13411,0.11311,0.10846,0.10269,0.09975,0.09226,0.09546,0.10353,0.0863,0.08647,0.0945,0.09677,0.0843,0.09172,0.09145,0.07016,0.09004,0.09058,0.09046,0.10613,0.07973,0.08355,0.06558,0.08679,0.08755,0.0863,0.10237,0.0998,0.09186,0.08727,0.10112,0.09055,0.08245,0.07335,0.08598,0.09782,0.11293,0.09069,0.08238,0.07828,0.09603,0.07384,0.09907,0.10577,0.10908,0.11418,0.13968,0.16042,0.18266,0.2462,0.2636,0.31162,0.30825,0.36082,0.36258,0.33226,0.30866,0.27159,0.21984,0.19418,0.14857,0.12557,0.12496,0.11146,0.08721,0.0826,0.08108,0.09358,0.07556,0.08872,0.09234,0.08377,0.09456,0.08122,0.08193,0.07411,0.09386,0.07583,0.0868,0.08239,0.07238,0.07662,0.09633,0.08432,0.07201,0.08495,0.08023,0.11603,0.08413,0.07727,0.09506,0.06224,0.08138,0.10017,0.07885,0.06247,0.07562,0.09391,0.08291,0.07054,0.07186,0.0729,0.08188,0.09143,0.08835,0.08217,0.06406,0.06388,0.08226,0.06325,0.07422,0.07929,0.06465,0.06555,0.07462,0.08037,0.08572,0.08094,0.07538,0.06695,0.06347,0.06884,0.07781,0.07685,0.05225,0.06515,0.08482,0.07333,0.05814,0.09563,0.07154,0.07544,0.07536,0.06745,0.06361,0.09187,0.08045,0.0784,0.0757,0.05967,0.08224,0.0608,0.07182,0.05236,0.0481,0.07146,0.05995,0.04791,0.07297,0.08853,0.08098,0.06428,0.0667,0.07525,0.07736,0.06247,0.06601,0.06844,0.07073,0.06363,0.06352,0.07829,0.07635,0.06879,0.08097,0.05969,0.07955,0.06851,0.09556,0.05069,0.05549,0.06634,0.0659,0.08119,0.05478,0.07028,0.07263,0.06782,0.05094,0.0744,0.07238,0.06586
lib_maltodextrin_r1,0,Raman785,1,library,maltodextrin,,,0.07314,0.08353,0.0816,0.07818,0.08297,0.09436,0.11434,0.12403,0.14062,0.15958,0.16615,0.20635,0.20923,0.22813,0.25747,0.25328,0.28075,0.31271,0.2963,0.33162,0.33799,0.31716,0.31739,0.31529,0.2856,0.28233,0.24834,0.2305,0.20713,0.19873,0.18801,0.15714,0.13057,0.11868,0.12167,0.12633,0.1038,0.0896,0.09616,0.08232,0.07814,0.07032,0.09259,0.05217,0.08494,0.07198,0.07234,0.09331,0.06928,0.0697,0.07532,0.0809,0.0725,0.06134,0.06858,0.09022,0.07832,0.08225,0.0753,0.07148,0.09502,0.07652,0.07413,0.08969,0.07166,0.0731,0.07931,0.07936,0.08211,0.10058,0.08318,0.05619,0.08338,0.08064,0.08085,0.08301,0.08041,0.08031,0.08711,0.0757,0.07178,0.11163,0.07436,0.08266,0.06683,0.08308,0.07271,0.07774,0.09274,0.08713,0.08822,0.06494,0.10114,0.07096,0.09828,0.09332,0.08899,0.07477,0.08495,0.07883,0.08393,0.08124,0.06345,0.09515,0.08322,0.07928,0.07862,0.0879,0.07548,0.06198,0.08873,0.09002,0.08307,0.07394,0.08683,0.07211,0.07772,0.08236,0.10108,0.09301,0.09779,0.1295,0.1237,0.13674,0.1461,0.1637,0.17101,0.22868,0.25174,0.27476,0.30407,0.33966,0.33989,0.36739,0.37321,0.40617,0.3934,0.35141,0.34053,0.33129,0.30115,0.26389,0.26748,0.22317,0.19906,0.16475,0.15982,0.13351,0.12295,0.12396,0.097,0.12788,0.11535,0.13085,0.1147,0.12654,0.13745,0.16786,0.18528,0.20206,0.22116,0.24525,0.2854,0.30968,0.33541,0.38402,0.39122,0.42436,0.4211,0.43804,0.43812,0.41853,0.44054,0.41006,0.39917,0.3606,0.3401,0.30506,0.29754,0.26022,0.23402,0.21917,0.19483,0.1713,0.1326,0.12575,0.11109,0.11911,0.10623,0.08499,0.09275,0.07654,0.08864,0.07095,0.09161,0.08927,0.10397,0.09295,0.07994,0.09212,0.06978,0.07398,0.0868,0.06176,0.08286,0.07291,0.08401,0.07211,0.08701,0.08284,0.09654,0.08248,0.0746,0.0797,0.0686,0.08597,0.08768,0.10635,0.0753,0.07871,0.0755,0.08324,0.0848,0.08031,0.09156,0.07439,0.09027,0.08565,0.08,0.07682,0.10409,0.0715,0.08249,0.07186,0.09706,0.08791,0.0721,0.08385,0.08114,0.07324,0.08582,0.08197,0.08007,0.07732,0.08188,0.0797,0.07782,0.07065,0.09382,0.06621,0.08081,0.07316,0.09378,0.06697,0.06847,0.06594,0.07869,0.06155,0.07031,0.05338,0.07713,0.06953,0.05947,0.08501,0.06089,0.08175,0.06658,0.08941,0.0807,0.06293,0.06365,0.07868,0.0698,0.07054,0.08475,0.06435,0.07255,0.08098,0.07933,0.06944,0.06795,0.05325,0.05987,0.07341,0.06865,0.07057,0.08001,0.05754,0.06685,0.07089,0.06176,0.08291,0.07747,0.07108,0.08592,0.05278,0.06748,0.07102,0.058,0.06994,0.05566,0.07342,0.07689,0.06882,0.04736,0.05601,0.07112,0.06548,0.03811,0.07727,0.066,0.05623,0.07416,0.05723,0.05691,0.06717,0.08119,0.0794,0.06655,0.06737,0.07045,0.04976,0.06856,0.05457,0.06488,0.05922,0.06629,0.06225,0.03014,0.06245,0.06687,0.0647,0.05901,0.07625,0.06274,0.06379,0.06965,0.06263,0.04982,0.03421,0.05362,0.06579,0.05445,0.06579,0.05906,0.07025,0.05422,0.06165,0.06183,0.07158,0.0393
lib_maltodextrin_r2,0,Raman785,2,library,maltodextrin,,,0.13361,0.12391,0.12291,0.13889,0.14688,0.14305,0.16018,0.18079,0.20195,0.2064,0.21705,0.23509,0.26114,0.27468,0.29763,0.30391,0.34428,0.33402,0.35184,0.36399,0.38554,0.35967,0.35128,0.34258,0.35145,0.29081,0.29104,0.30936,0.26858,0.23255,0.22506,0.2094,0.20214,0.1989,0.17316,0.15453,0.16022,0.13937,0.13141,0.1207,0.12016,0.12511,0.12116,0.13538,0.1199,0.10034,0.12139,0.12728,0.1262,0.14775,0.12675,0.14205,0.13342,0.13355,0.126,0.13414,0.12745,0.13717,0.13348,0.12676,0.14649,0.1412,0.13281,0.1391,0.13716,0.12967,0.1394,0.13283,0.12993,0.12349,0.13647,0.14606,0.13377,0.16625,0.15022,0.15473,0.14995,0.16084,0.13475,0.11732,0.13877,0.14911,0.14534,0.14687,0.15761,0.1485,0.15295,0.13584,0.13456,0.16192,0.14706,0.14008,0.14035,0.14033,0.13998,0.13449,0.13256,0.14585,0.1421,0.15934,0.15142,0.15137,0.1415,0.14794,0.15297,0.15673,0.14883,0.16076,0.15026,0.15832,0.14573,0.13277,0.15391,0.15891,0.15802,0.15737,0.15096,0.14829,0.16465,0.16738,0.16306,0.18733,0.17743,0.19332,0.2316,0.24334,0.25373,0.28568,0.31194,0.31151,0.35808,0.39284,0.41236,0.43583,0.45569,0.445,0.43396,0.43691,0.41312,0.39242,0.36505,0.36242,0.28929,0.28057,0.25938,0.22957,0.22783,0.21209,0.19225,0.1823,0.17946,0.17818,0.18002,0.18331,0.19492,0.20997,0.20306,0.22071,0.23247,0.28009,0.29785,0.32629,0.33415,0.39397,0.39718,0.43051,0.46021,0.46085,0.47959,0.49572,0.52545,0.49675,0.50075,0.48222,0.4546,0.418,0.41274,0.35982,0.36293,0.32151,0.26688,0.26579,0.25392,0.22582,0.21887,0.18938,0.17346,0.17667,0.17049,0.16111,0.17261,0.14866,0.15794,0.15376,0.13625,0.1377,0.15032,0.16948,0.15505,0.13165,0.1175,0.15566,0.14711,0.15923,0.13877,0.15083,0.15554,0.14235,0.14231,0.14656,0.15148,0.14564,0.15525,0.14062,0.13168,0.16445,0.13835,0.13843,0.14316,0.1442,0.13936,0.14275,0.14222,0.14406,0.11842,0.16049,0.13989,0.14685,0.12694,0.13568,0.13333,0.15442,0.13352,0.14662,0.13219,0.13382,0.13554,0.15049,0.11839,0.12377,0.1171,0.12533,0.12772,0.11932,0.12775,0.11457,0.12416,0.14596,0.13761,0.13507,0.12278,0.11705,0.12063,0.13119,0.13991,0.13985,0.13673,0.12265,0.12986,0.12599,0.12678,0.14875,0.11035,0.12521,0.13925,0.11206,0.11985,0.12058,0.12321,0.10906,0.13054,0.13082,0.12881,0.11459,0.12372,0.11342,0.11865,0.14307,0.12206,0.10878,0.12202,0.10625,0.10733,0.10846,0.11398,0.12735,0.12331,0.12231,0.10563,0.11076,0.11844,0.10898,0.11202,0.11876,0.1104,0.1157,0.1289,0.1108,0.12385,0.11719,0.09646,0.09965,0.10188,0.1012,0.10839,0.11497,0.12092,0.11098,0.10298,0.08615,0.10722,0.10631,0.10741,0.12618,0.11706,0.11204,0.09411,0.09814,0.10185,0.09459,0.10688,0.10498,0.09007,0.11246,0.10105,0.08228,0.09859,0.10146,0.10791,0.09435,0.10555,0.09868,0.0832,0.08174,0.10666,0.10694,0.09776,0.10903,0.10419,0.1,0.0836,0.10958,0.09718,0.08751,0.08424,0.09304,0.07331,0.09658,0.0796,0.0924,0.08304
lib_maltodextrin_r3,0,Raman785,3,library,maltodextrin,,,0.21863,0.24311,0.24299,0.23879,0.24249,0.26029,0.27402,0.26721,0.28946,0.28865,0.33644,0.3656,0.36641,0.36535,0.41331,0.41765,0.43757,0.46525,0.47292,0.47295,0.47453,0.4811,0.47029,0.45391,0.4489,0.44011,0.41747,0.3865,0.39153,0.33558,0.34077,0.32269,0.31573,0.30712,0.30198,0.2747,0.28058,0.27115,0.24795,0.28112,0.24573,0.26308,0.26215,0.26082,0.25326,0.26686,0.24354,0.25916,0.25492,0.25458,0.26647,0.2738,0.26044,0.27253,0.26607,0.2575,0.26469,0.26512,0.26198,0.27406,0.27295,0.25551,0.25571,0.27728,0.26567,0.27764,0.28321,0.26347,0.26283,0.29446,0.28199,0.27503,0.27062,0.27559,0.26952,0.29098,0.27509,0.2945,0.28025,0.27451,0.28136,0.26006,0.28576,0.28592,0.29652,0.27005,0.30117,0.29838,0.30331,0.29087,0.30214,0.27788,0.3003,0.28603,0.2778,0.28362,0.29547,0.28771,0.30024,0.27892,0.27624,0.31174,0.30755,0.2885,0.29944,0.31013,0.30458,0.30233,0.31224,0.29444,0.30252,0.30253,0.30053,0.30001,0.30546,0.30617,0.29685,0.31747,0.30268,0.31445,0.31945,0.33378,0.31928,0.36254,0.36322,0.40191,0.40791,0.43803,0.48559,0.49261,0.53629,0.55082,0.57745,0.59166,0.61066,0.59927,0.59266,0.58523,0.57528,0.54956,0.53271,0.51053,0.45924,0.43425,0.40203,0.41452,0.37859,0.36536,0.33587,0.34149,0.34213,0.33314,0.348,0.36478,0.33943,0.34877,0.36321,0.40399,0.41719,0.42701,0.45278,0.47029,0.5147,0.52982,0.55831,0.59267,0.60763,0.62627,0.64307,0.65976,0.67113,0.65724,0.65016,0.62536,0.60075,0.59101,0.56009,0.51643,0.49912,0.47378,0.443,0.43837,0.40618,0.39559,0.36769,0.35173,0.32989,0.32996,0.32249,0.30284,0.30661,0.30533,0.31009,0.30859,0.29664,0.30596,0.29849,0.27083,0.29399,0.309,0.28081,0.2982,0.31237,0.28837,0.29545,0.28183,0.31059,0.29291,0.29364,0.30613,0.29861,0.28264,0.28432,0.29512,0.28331,0.26774,0.28774,0.27282,0.30313,0.28807,0.27658,0.27796,0.26536,0.27257,0.29302,0.30446,0.28869,0.27977,0.27564,0.26965,0.29907,0.27583,0.29811,0.29011,0.25799,0.26796,0.26714,0.27223,0.27162,0.30138,0.25983,0.27207,0.27511,0.26809,0.26837,0.25597,0.26663,0.26982,0.26712,0.26868,0.27622,0.27058,0.26104,0.24772,0.25254,0.25544,0.25925,0.25103,0.24554,0.24209,0.23654,0.24977,0.23648,0.24024,0.24536,0.23499,0.25119,0.25384,0.23998,0.24014,0.24892,0.23543,0.23377,0.25258,0.24848,0.24466,0.24691,0.21689,0.24921,0.23859,0.24378,0.23654,0.23215,0.2279,0.23331,0.22551,0.23629,0.21072,0.22015,0.24499,0.21726,0.22011,0.21264,0.22379,0.21204,0.21651,0.22582,0.20931,0.22314,0.22244,0.22373,0.21354,0.24351,0.2206,0.21226,0.19921,0.20595,0.22014,0.22193,0.21857,0.21059,0.18913,0.20551,0.18718,0.18367,0.19989,0.20169,0.20444,0.19934,0.19863,0.18243,0.19018,0.17511,0.16911,0.19551,0.17953,0.19474,0.19606,0.17775,0.18881,0.17526,0.16237,0.18103,0.17406,0.17334,0.19646,0.16042,0.17849,0.16369,0.18169,0.17458,0.17437,0.16011,0.16089,0.17681,0.18423,0.17032,0.15554,0.1669,0.18679,0.16254
unk_1,0,Raman785,1,unknown,unknown,,,0.14015,0.1478,0.15509,0.15353,0.14256,0.14279,0.16143,0.16413,0.16799,0.17157,0.15554,0.15174,0.16902,0.16329,0.14699,0.14713,0.16304,0.157,0.15915,0.17079,0.1683,0.17465,0.14267,0.16516,0.17021,0.14863,0.15798,0.15676,0.16601,0.1827,0.18727,0.15952,0.17355,0.17449,0.17339,0.17412,0.17377,0.18822,0.18297,0.16196,0.18115,0.18281,0.17787,0.18718,0.17953,0.18263,0.18679,0.17337,0.20482,0.21585,0.23236,0.28374,0.3004,0.31805,0.32875,0.34346,0.34541,0.31226,0.29859,0.28245,0.23032,0.2285,0.20896,0.20094,0.19037,0.1929,0.20887,0.1857,0.19179,0.20907,0.18401,0.2081,0.19047,0.20871,0.19885,0.19489,0.19161,0.18648,0.19393,0.20259,0.19392,0.19132,0.1924,0.20526,0.208,0.18949,0.18988,0.20076,0.21264,0.20921,0.18562,0.19588,0.18959,0.19659,0.21567,0.18575,0.18414,0.19239,0.20389,0.20756,0.20605,0.21376,0.18469,0.21451,0.19981,0.21054,0.19029,0.20386,0.20645,0.21592,0.18878,0.2052,0.19429,0.21045,0.21341,0.21874,0.20075,0.2156,0.22015,0.19032,0.21308,0.21674,0.20476,0.19542,0.21714,0.19589,0.20626,0.2078,0.21551,0.20224,0.2018,0.20976,0.21896,0.23189,0.19405,0.21239,0.19847,0.21961,0.20012,0.20284,0.20377,0.20897,0.21309,0.21525,0.21338,0.21359,0.28029,0.39194,0.63288,0.92474,1.18058,1.25989,1.09362,0.77373,0.49062,0.31191,0.26121,0.20756,0.20401,0.21564,0.19112,0.22234,0.21575,0.21842,0.21491,0.20762,0.1985,0.20762,0.19915,0.21364,0.21671,0.22327,0.21208,0.21745,0.20508,0.2031,0.21245,0.20367,0.21004,0.19654,0.19814,0.21311,0.21541,0.19647,0.21281,0.20055,0.2058,0.21956,0.21322,0.21285,0.22372,0.19732,0.19423,0.19659,0.20098,0.17968,0.20086,0.19718,0.21015,0.20552,0.19567,0.20725,0.20825,0.19299,0.1954,0.17773,0.18872,0.19184,0.19963,0.18906,0.193,0.1855,0.19908,0.20123,0.20059,0.19628,0.18034,0.18913,0.19084,0.19045,0.1948,0.20787,0.20129,0.18285,0.19906,0.19822,0.18329,0.19937,0.18881,0.17584,0.19401,0.19114,0.1862,0.18761,0.1803,0.19309,0.18604,0.18763,0.18902,0.17338,0.17546,0.16178,0.18899,0.17069,0.18359,0.18543,0.1872,0.16859,0.17942,0.19621,0.19609,0.18826,0.18119,0.16719,0.18385,0.17451,0.18111,0.16859,0.1726,0.17538,0.17561,0.17748,0.16615,0.15791,0.18698,0.19502,0.17399,0.15566,0.16055,0.17276,0.18045,0.15429,0.16455,0.16712,0.17318,0.16047,0.14763,0.16885,0.17325,0.16055,0.16483,0.15547,0.15128,0.16969,0.16326,0.16472,0.17604,0.15282,0.16187,0.17525,0.1772,0.20326,0.20986,0.22174,0.25525,0.28099,0.31329,0.33031,0.34892,0.35893,0.35737,0.34723,0.34453,0.31904,0.3155,0.28603,0.25291,0.22611,0.20018,0.17081,0.16734,0.16002,0.15369,0.15074,0.15158,0.13293,0.13126,0.15132,0.14352,0.13106,0.13314,0.13247,0.13313,0.13686,0.15681,0.13269,0.13675,0.11259,0.12394,0.14042,0.13149,0.08291,0.13288,0.14465,0.10485,0.13935,0.09577,0.13221,0.12209,0.09802,0.12533,0.13172,0.1232,0.12476,0.11415,0.099,0.12389,0.13382,0.12698,0.09516,0.09925
unk_2,0,Raman785,1,unknown,unknown,,,0.13163,0.14867,0.15283,0.13526,0.158,0.15691,0.14608,0.15051,0.1334,0.15024,0.15431,0.1566,0.15937,0.15871,0.15431,0.16051,0.16837,0.15811,0.16298,0.1587,0.15488,0.16739,0.15747,0.1678,0.16519,0.16292,0.16381,0.16326,0.15326,0.17507,0.17526,0.18336,0.1717,0.1984,0.1784,0.17655,0.17889,0.17547,0.21263,0.21609,0.21408,0.20549,0.23969,0.226,0.23561,0.26237,0.26525,0.29232,0.28546,0.29272,0.30533,0.30879,0.3413,0.37333,0.3866,0.40039,0.41056,0.44373,0.47305,0.48903,0.50838,0.52072,0.55914,0.54051,0.59641,0.63819,0.61626,0.64652,0.66497,0.67488,0.70249,0.70749,0.7128,0.72554,0.72173,0.755,0.72399,0.72136,0.69693,0.70101,0.7082,0.67166,0.6499,0.6566,0.65331,0.62209,0.59931,0.56425,0.5669,0.54813,0.50263,0.51071,0.48612,0.46924,0.4437,0.42498,0.40497,0.39062,0.34684,0.33409,0.32682,0.31901,0.31136,0.29894,0.29609,0.28663,0.2576,0.26079,0.24286,0.24319,0.24277,0.20063,0.25462,0.23253,0.21777,0.24033,0.21531,0.21819,0.2104,0.23047,0.21181,0.21347,0.22331,0.21656,0.20973,0.22104,0.23507,0.21269,0.21056,0.21102,0.22239,0.20328,0.2065,0.19394,0.19752,0.20424,0.20891,0.22722,0.21495,0.20053,0.20242,0.2235,0.22107,0.21013,0.20547,0.22117,0.20002,0.2139,0.23273,0.21209,0.20234,0.21437,0.206,0.19776,0.22088,0.22937,0.22304,0.23584,0.25093,0.28881,0.3232,0.36358,0.4389,0.46551,0.49478,0.50792,0.49656,0.45631,0.4221,0.37018,0.33219,0.29251,0.25529,0.22469,0.22578,0.21412,0.19273,0.21434,0.21771,0.23271,0.20314,0.2014,0.19546,0.20842,0.20813,0.19776,0.20389,0.18561,0.21441,0.19718,0.20428,0.19009,0.19826,0.20101,0.20341,0.20803,0.21732,0.20995,0.21194,0.21646,0.20737,0.20098,0.21111,0.21141,0.20682,0.18543,0.19121,0.19651,0.20669,0.1983,0.20951,0.21091,0.20936,0.20902,0.1862,0.20099,0.18967,0.17884,0.19204,0.20773,0.19866,0.21379,0.19494,0.2172,0.19405,0.19204,0.17856,0.20199,0.18539,0.18347,0.20227,0.20055,0.1998,0.18437,0.18698,0.18639,0.1943,0.2015,0.18913,0.20468,0.19338,0.18153,0.19163,0.19987,0.20225,0.19359,0.18143,0.17846,0.18583,0.19625,0.18329,0.18293,0.17833,0.17396,0.17504,0.17744,0.19269,0.17252,0.17486,0.16508,0.18031,0.18621,0.18819,0.17337,0.17648,0.16859,0.17479,0.17204,0.16887,0.16904,0.17156,0.16348,0.16337,0.18224,0.19164,0.15186,0.15067,0.15306,0.16096,0.17031,0.16662,0.16178,0.16471,0.15185,0.15568,0.15727,0.14097,0.14455,0.14429,0.16249,0.15665,0.15964,0.13448,0.13738,0.15818,0.1231,0.16247,0.13132,0.15732,0.13934,0.15805,0.17897,0.12349,0.1525,0.1268,0.14848,0.14147,0.14417,0.13308,0.14933,0.12637,0.1526,0.13542,0.12481,0.15412,0.13682,0.1559,0.14837,0.1275,0.13396,0.13535,0.14873,0.13718,0.13389,0.12946,0.13029,0.14317,0.1295,0.12484,0.12248,0.12192,0.11278,0.11863,0.14046,0.12275,0.1312,0.09871,0.12852,0.12056,0.11671,0.12268,0.11526,0.09832,0.12813,0.12069,0.11448,0.10731,0.10163,0.11792,0.13117,0.11643
unk_3,0,Raman785,1,unknown,unknown,,,0.15264,0.1517,0.1631,0.16322,0.13055,0.16976,0.15889,0.16008,0.17857,0.19148,0.17856,0.21018,0.2193,0.2208,0.22017,0.2271,0.25442,0.23522,0.26579,0.25673,0.25894,0.24258,0.25625,0.26033,0.23594,0.24831,0.23745,0.22289,0.2305,0.19268,0.21129,0.21769,0.21026,0.19488,0.18387,0.17719,0.1779,0.18438,0.16944,0.18413,0.19731,0.17509,0.16223,0.16237,0.18386,0.17587,0.18568,0.17206,0.18621,0.17686,0.17166,0.17552,0.16811,0.20053,0.18022,0.18817,0.1769,0.17229,0.18101,0.19489,0.1792,0.17622,0.187,0.19096,0.20291,0.19052,0.19393,0.19518,0.17695,0.17934,0.17862,0.17662,0.19837,0.19379,0.19215,0.16589,0.21283,0.18378,0.21369,0.17912,0.18593,0.17245,0.19853,0.18432,0.18502,0.18244,0.20434,0.20408,0.21548,0.1909,0.20033,0.18562,0.19554,0.19407,0.20427,0.21015,0.21024,0.19225,0.211,0.20091,0.19843,0.19491,0.19672,0.22005,0.20601,0.20498,0.2009,0.22829,0.2172,0.19737,0.20257,0.20465,0.21144,0.20943,0.23269,0.26209,0.26172,0.29663,0.33628,0.36237,0.398,0.38235,0.3621,0.32343,0.29772,0.26745,0.28032,0.27639,0.28949,0.2833,0.29429,0.30766,0.32395,0.30742,0.33401,0.32806,0.31627,0.30168,0.31424,0.32108,0.31289,0.28844,0.29311,0.26208,0.26196,0.23903,0.23092,0.2233,0.20495,0.23532,0.21426,0.21221,0.20935,0.22172,0.24648,0.25878,0.23635,0.27317,0.30044,0.36415,0.40517,0.44008,0.47799,0.48362,0.49556,0.46142,0.43339,0.3728,0.37797,0.38559,0.3692,0.33114,0.34052,0.3248,0.34574,0.32156,0.30088,0.29771,0.2599,0.2639,0.26441,0.26258,0.22574,0.23343,0.24052,0.21015,0.2222,0.21669,0.23008,0.2082,0.20688,0.21567,0.1927,0.20286,0.19979,0.20238,0.21176,0.20838,0.19496,0.20345,0.21663,0.19306,0.20248,0.19875,0.19971,0.18369,0.21376,0.19159,0.20935,0.19277,0.19292,0.21208,0.21167,0.20722,0.2185,0.21689,0.23225,0.24194,0.2767,0.28096,0.32158,0.31813,0.32929,0.33622,0.32019,0.30151,0.28446,0.27257,0.23701,0.22526,0.21318,0.21194,0.19565,0.21044,0.18812,0.18565,0.18523,0.17299,0.17516,0.17965,0.19577,0.18355,0.1785,0.19507,0.19778,0.19428,0.17265,0.17567,0.19821,0.18192,0.17627,0.18557,0.18789,0.17153,0.17886,0.18535,0.1639,0.17504,0.16319,0.1639,0.16751,0.17732,0.16872,0.16486,0.17125,0.15866,0.1738,0.18021,0.16749,0.16564,0.17078,0.14613,0.17116,0.19618,0.16331,0.16271,0.1707,0.17908,0.17174,0.13979,0.16634,0.18578,0.16564,0.14722,0.16094,0.14466,0.16914,0.1466,0.15625,0.14627,0.14873,0.16839,0.15391,0.14662,0.1401,0.13217,0.13771,0.15068,0.15869,0.14262,0.16326,0.13971,0.14384,0.13475,0.15479,0.16519,0.12592,0.13941,0.15485,0.12706,0.14652,0.1524,0.14029,0.14434,0.15578,0.15698,0.1362,0.13083,0.12628,0.14458,0.12042,0.13094,0.13383,0.1271,0.13426,0.15644,0.12891,0.1251,0.13895,0.13052,0.14186,0.12845,0.12365,0.10764,0.10861,0.10579,0.12729,0.10921,0.13473,0.12182,0.11843,0.11178,0.12579,0.11087,0.10841,0.1295,0.12052,0.11398,0.1221,0.11513,0.11823
mix_demo,0,Raman785,1,mixture,mixture,0.62,0.34,0.09651,0.12668,0.10182,0.12521,0.11028,0.12331,0.12626,0.13472,0.11985,0.14468,0.15863,0.14431,0.1543,0.15589,0.16471,0.19194,0.17078,0.17647,0.20591,0.19862,0.21471,0.18864,0.19262,0.18164,0.18545,0.16115,0.18001,0.17033,0.16042,0.1512,0.14069,0.15207,0.12898,0.14254,0.15577,0.14838,0.11784,0.12774,0.13835,0.13523,0.12447,0.14555,0.14552,0.16936,0.15294,0.17309,0.15534,0.16501,0.19048,0.20275,0.19168,0.21152,0.20814,0.23146,0.25354,0.25327,0.26807,0.2726,0.29108,0.32903,0.32618,0.32943,0.35018,0.35096,0.37001,0.40643,0.4034,0.43492,0.4253,0.43508,0.44985,0.42345,0.45759,0.459,0.47682,0.47256,0.45103,0.47273,0.45415,0.44276,0.44745,0.43586,0.4445,0.42268,0.41055,0.40591,0.39556,0.37966,0.37281,0.35124,0.33323,0.33035,0.30356,0.29154,0.29548,0.28196,0.2536,0.23764,0.24589,0.22897,0.21449,0.19914,0.19739,0.18623,0.17283,0.17443,0.17008,0.15853,0.16674,0.16653,0.15976,0.14417,0.15179,0.14272,0.12767,0.14524,0.13887,0.16268,0.15116,0.14414,0.14065,0.14177,0.13303,0.1297,0.14357,0.15826,0.1453,0.16956,0.18193,0.19204,0.19726,0.20947,0.2379,0.23329,0.22486,0.23109,0.23021,0.21905,0.21306,0.22309,0.19979,0.18336,0.18408,0.17626,0.14669,0.15838,0.16105,0.15173,0.16564,0.15319,0.15368,0.16683,0.14724,0.14346,0.15546,0.1464,0.16266,0.16926,0.19966,0.21573,0.2484,0.2972,0.33085,0.3488,0.38106,0.39948,0.40351,0.4049,0.38161,0.36336,0.33239,0.30972,0.28277,0.26026,0.25379,0.23885,0.22167,0.21971,0.17319,0.18393,0.17458,0.1807,0.17727,0.14036,0.16322,0.13256,0.12887,0.13473,0.13134,0.11695,0.13751,0.13665,0.13952,0.12399,0.12744,0.12715,0.14403,0.09856,0.11713,0.12657,0.11549,0.1236,0.12476,0.11929,0.12635,0.13183,0.12303,0.1207,0.12341,0.12482,0.10733,0.12917,0.11987,0.13682,0.12994,0.13555,0.11356,0.11977,0.11087,0.11605,0.13543,0.13759,0.11104,0.11774,0.12136,0.12777,0.10411,0.10547,0.12619,0.10159,0.1164,0.12227,0.12459,0.1188,0.11634,0.12822,0.1224,0.11285,0.11947,0.12269,0.13581,0.12625,0.1303,0.11181,0.11243,0.11508,0.11575,0.10686,0.12936,0.12528,0.11314,0.12237,0.11792,0.12264,0.10174,0.11347,0.11409,0.12064,0.10786,0.08888,0.1028,0.11274,0.11373,0.12966,0.11838,0.11309,0.11168,0.12278,0.10556,0.10978,0.10002,0.12482,0.11428,0.103,0.10865,0.11461,0.10672,0.1036,0.10212,0.11177,0.10451,0.10327,0.09416,0.10801,0.10513,0.10344,0.10835,0.11355,0.1128,0.11102,0.10279,0.09923,0.09417,0.09838,0.10422,0.09586,0.10439,0.09348,0.08623,0.11074,0.0975,0.09358,0.09071,0.10559,0.11481,0.09673,0.09831,0.11088,0.08176,0.11924,0.12371,0.07191,0.0977,0.09118,0.09543,0.09359,0.1056,0.09605,0.08777,0.09486,0.10143,0.08415,0.07923,0.11532,0.08867,0.09342,0.09052,0.09968,0.11164,0.09941,0.07813,0.07918,0.08874,0.08975,0.07492,0.10004,0.08069,0.10619,0.10917,0.08819,0.09478,0.07778,0.09604,0.08453,0.07836,0.08605,0.084,0.08511,0.09758,0.07994,0.10486
mix_exercise,0,Raman785,1,mixture,mixture,,,0.10257,0.13375,0.10969,0.12271,0.12138,0.12721,0.14393,0.15715,0.16128,0.15351,0.16845,0.16854,0.17667,0.18552,0.20282,0.21932,0.20636,0.2323,0.25349,0.25315,0.23754,0.22868,0.24438,0.21177,0.23952,0.21842,0.23007,0.18515,0.20213,0.18272,0.17174,0.15348,0.16624,0.16123,0.16765,0.15635,0.13788,0.13236,0.12823,0.15703,0.13568,0.16013,0.1531,0.15427,0.1516,0.16551,0.16353,0.16529,0.18175,0.17968,0.17997,0.20129,0.20246,0.21627,0.21555,0.21709,0.25599,0.26385,0.26721,0.26464,0.27574,0.30279,0.29556,0.30271,0.33741,0.32748,0.35441,0.33092,0.35758,0.37984,0.36529,0.3673,0.36222,0.38764,0.37096,0.37847,0.36673,0.3932,0.38667,0.37126,0.35549,0.37693,0.3516,0.33712,0.32277,0.32557,0.30863,0.3077,0.31184,0.29091,0.2884,0.26779,0.26593,0.27491,0.24287,0.24114,0.21294,0.23548,0.20763,0.21548,0.19678,0.19766,0.19228,0.17554,0.18597,0.1629,0.17415,0.15878,0.16289,0.15186,0.15443,0.14097,0.1523,0.1487,0.14162,0.14558,0.14699,0.13949,0.14495,0.13752,0.17328,0.16378,0.16062,0.17585,0.15226,0.15614,0.18909,0.19214,0.2175,0.23639,0.25782,0.25053,0.27737,0.28236,0.29129,0.28455,0.3017,0.28128,0.29126,0.2542,0.25628,0.23401,0.21691,0.21264,0.19058,0.15573,0.14227,0.17805,0.17192,0.17457,0.15329,0.13993,0.14967,0.15669,0.15119,0.18488,0.2011,0.19785,0.21227,0.22425,0.28281,0.30222,0.33401,0.37487,0.39121,0.41626,0.40727,0.42908,0.40538,0.40093,0.36789,0.34896,0.31968,0.32222,0.28975,0.27818,0.2679,0.25413,0.22117,0.22174,0.21859,0.19448,0.18488,0.17895,0.16766,0.15154,0.15568,0.1613,0.13715,0.14105,0.1288,0.15038,0.13882,0.13033,0.13909,0.12563,0.12695,0.12239,0.12608,0.15012,0.12903,0.14085,0.12766,0.14346,0.12545,0.13594,0.13592,0.15851,0.14137,0.12897,0.14158,0.15594,0.11809,0.12578,0.13166,0.12968,0.13034,0.12044,0.14733,0.11888,0.13091,0.14318,0.13222,0.12253,0.11863,0.12962,0.12414,0.13356,0.11728,0.13921,0.13203,0.13255,0.14485,0.12584,0.12274,0.12429,0.12762,0.12841,0.12078,0.10863,0.12302,0.13199,0.12952,0.13991,0.14723,0.13503,0.14102,0.13461,0.1438,0.14496,0.13571,0.13301,0.12364,0.12503,0.12181,0.13114,0.11747,0.1225,0.10059,0.14111,0.12141,0.12791,0.13766,0.12582,0.13699,0.10986,0.14598,0.1326,0.12355,0.13544,0.10922,0.11774,0.13479,0.1014,0.1254,0.12483,0.13892,0.10312,0.13476,0.11278,0.11362,0.10559,0.10664,0.12127,0.10582,0.12406,0.10973,0.10394,0.1105,0.11886,0.11242,0.11525,0.12157,0.12514,0.11092,0.10329,0.11419,0.11784,0.09823,0.10364,0.10506,0.10987,0.10404,0.11843,0.09501,0.09496,0.10712,0.10841,0.09532,0.1035,0.10526,0.10504,0.11019,0.10176,0.12285,0.11266,0.09941,0.10812,0.10812,0.11873,0.10028,0.11366,0.10611,0.12031,0.118,0.10677,0.10188,0.10927,0.1021,0.12042,0.1041,0.09897,0.09813,0.0903,0.1077,0.09365,0.11253,0.08833,0.10224,0.11218,0.07886,0.12028,0.09555,0.10093,0.10494,0.1062,0.10139,0.10834,0.10327,0.1154,0.10273
"""

In [ ]:
def _is_number(text):
    try:
        float(text)
        return True
    except (TypeError, ValueError):
        return False


def load_spectra(source=None):
    """讀取寬格式光譜 CSV（見上方「光譜資料格式」說明）。

    Parameters
    ----------
    source : str, path, file-like, or None
        - None：退回本筆記本內嵌的模擬光譜字串 SAMPLE_CSV（模擬光譜，非真實量測）。
        - 本機/雲端硬碟路徑，或 http(s) URL（例如已發布的 Google 試算表 CSV 連結）：
          直接用 pandas.read_csv 讀取。

    Returns
    -------
    pandas.DataFrame
        寬格式：中繼資料欄位（欄名非數字）+ 光譜欄位（欄名為數字，代表 nm 或 cm-1）。
    """
    if source is None:
        return pd.read_csv(io.StringIO(SAMPLE_CSV))
    return pd.read_csv(source)


def split_spectra(df):
    """把 load_spectra() 讀進來的寬格式 DataFrame 拆成 (meta, X, axis)。

    自動判斷兩種輸入：
    - 一般格式：欄名可轉成浮點數的欄位＝光譜欄位；其餘欄位＝中繼資料 (meta)。
    - SpectraView「rows」匯出格式：第一個欄名以 "x:" 開頭，代表該欄其實是
      「樣品名稱」，其餘欄名本身就是 x 值；此時 meta 只有一欄 sample_id
      （由第一欄改名而來），其他中繼資料需自行用 sample_id 合併(merge)。

    Returns
    -------
    meta : pandas.DataFrame   中繼資料（含 sample_id 或其他非光譜欄位）
    X    : numpy.ndarray, shape (n_spectra, n_points)   光譜強度矩陣
    axis : numpy.ndarray, shape (n_points,)             x 軸，已由小到大排序
    """
    cols = list(df.columns)

    if isinstance(cols[0], str) and cols[0].startswith("x:"):
        meta = df.iloc[:, [0]].rename(columns={cols[0]: "sample_id"}).reset_index(drop=True)
        spec_cols = cols[1:]
    else:
        spec_cols = [c for c in cols if _is_number(c)]
        meta_cols = [c for c in cols if c not in spec_cols]
        meta = df[meta_cols].reset_index(drop=True)

    axis = np.array([float(c) for c in spec_cols])
    X = df[spec_cols].to_numpy(dtype=float)

    order = np.argsort(axis)
    axis = axis[order]
    X = X[:, order]
    return meta, X, axis


def plot_spectra(axis, X, labels, ax=None, title=None, xlabel="x", ylabel="Intensity (a.u.)"):
    """畫出多條光譜，依 labels 分色，圖例只顯示每個 label 一次。"""
    if ax is None:
        _, ax = plt.subplots(figsize=(9, 5))
    labels = list(labels)
    unique_labels = sorted(set(labels), key=labels.index)
    cmap = plt.get_cmap("tab10")
    color_of = {lab: cmap(i % 10) for i, lab in enumerate(unique_labels)}
    seen = set()
    for i in range(X.shape[0]):
        lab = labels[i]
        show_label = lab if lab not in seen else None
        seen.add(lab)
        ax.plot(axis, X[i], color=color_of[lab], linewidth=1.3, alpha=0.85, label=show_label)
    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)
    if title:
        ax.set_title(title)
    ax.legend(loc="best", fontsize=8)
    return ax

In [ ]:
df_raw = load_spectra()
meta, X, axis = split_spectra(df_raw)
print(f"讀入 {len(meta)} 條光譜，x 軸範圍 {axis.min():.0f}-{axis.max():.0f} cm-1，"
      f"共 {len(axis)} 個資料點。")
meta.head(10)

## 2. Raman 前處理：尖峰移除 + ALS 基線移除 (Despiking & ALS Baseline)

原始 Raman 光譜常見兩個問題：
- **宇宙射線尖峰 (cosmic spikes)**：偵測器偶爾被高能宇宙射線擊中，產生極窄、極高的假訊號。
- **螢光基線 (fluorescence baseline)**：有機雜質/染料被雷射激發，疊加一個寬廣的背景。

### 尖峰移除：中值濾波 + 修改式 Z-score

先用中值濾波估計「局部應該長怎樣」，再看每個點偏離中值濾波結果多遠（用穩健的 MAD 估計取代標準差），
偏離太多的點視為尖峰，用中值濾波值取代。

### ALS 基線移除 (Asymmetric Least Squares, Eilers 2005)

ALS 用兩個參數擬合一條平滑基線：
- **λ (lambda)**：基線的平滑度，λ 越大基線越平滑、越不會跟著波峰起伏；λ 太小則會把真實波峰也當成基線的一部分扣掉。
- **p**：不對稱性權重，p 越小代表越不信任「高於目前基線估計」的點（也就是波峰），只用低於基線的點來擬合下一輪基線。

airPLS（adaptive iteratively reweighted penalized least squares）是 ALS 的一個變體，
用波峰以下的殘差自動調整權重，只需要調 λ 一個參數，本節後段會示範。

In [ ]:
def despike(y, window=7, z_thresh=6.0):
    """中值濾波 + 修改式 Z-score 尖峰偵測。回傳 (去尖峰後的光譜, 尖峰位置遮罩)。"""
    y = np.asarray(y, dtype=float)
    med = median_filter(y, size=window, mode="nearest")
    diff = y - med
    mad = np.median(np.abs(diff - np.median(diff))) + 1e-12
    z = 0.6745 * diff / mad
    mask = np.abs(z) > z_thresh
    y_clean = y.copy()
    y_clean[mask] = med[mask]
    return y_clean, mask


def als_baseline(y, lam=1e5, p=0.01, niter=10):
    """Eilers (2005) 不對稱最小平方基線。lam=平滑度，p=不對稱性權重。"""
    y = np.asarray(y, dtype=float)
    L = len(y)
    D = sparse.diags([1, -2, 1], [0, -1, -2], shape=(L, L - 2))
    D = lam * D.dot(D.T)
    w = np.ones(L)
    z = y.copy()
    for _ in range(niter):
        W = sparse.spdiags(w, 0, L, L)
        Z = (W + D).tocsc()
        z = spsolve(Z, w * y)
        w = p * (y > z) + (1 - p) * (y < z)
    return z


def airpls_baseline(y, lam=1e5, itermax=15):
    """airPLS (Zhang et al., 2010)：只需要調 lam，權重由殘差自動調整。"""
    y = np.asarray(y, dtype=float)
    L = len(y)
    D = sparse.diags([1, -2, 1], [0, -1, -2], shape=(L, L - 2))
    D = lam * D.dot(D.T)
    w = np.ones(L)
    z = y.copy()
    for i in range(1, itermax + 1):
        W = sparse.spdiags(w, 0, L, L)
        Z = (W + D).tocsc()
        z = spsolve(Z, w * y)
        d = y - z
        neg = d < 0
        dssn = np.abs(d[neg].sum()) if neg.any() else 1e-9
        if dssn < 0.001 * np.abs(y).sum():
            break
        w[~neg] = 0
        w[neg] = np.exp(i * np.abs(d[neg]) / dssn)
        if neg.any():
            w[0] = w[neg].max()
            w[-1] = w[0]
    return z

In [ ]:
# 用 unk_1 (含尖峰＋螢光基線的示意光譜) 示範 despike + ALS 流程
# 這裡另外疊加兩個人工尖峰，方便清楚看到 despike 的效果
demo_idx = meta.index[meta["sample_id"] == "unk_1"][0]
y_demo_raw = X[demo_idx].copy()
spike_positions = [460, 1300]
rng_demo = np.random.default_rng(0)
for pos in spike_positions:
    i = int(np.argmin(np.abs(axis - pos)))
    y_demo_raw[i] += rng_demo.uniform(0.6, 0.9)

y_desp, spike_mask = despike(y_demo_raw, window=7, z_thresh=6.0)
baseline_als = als_baseline(y_desp, lam=1e6, p=0.01, niter=10)
y_clean_als = y_desp - baseline_als

print(f"偵測到 {spike_mask.sum()} 個尖峰，位置(cm-1)：{axis[spike_mask]}")

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
axes[0].plot(axis, y_demo_raw, color="tab:orange", label="raw (spikes + baseline)")
axes[0].plot(axis, y_desp, color="tab:blue", label="after despike")
axes[0].scatter(axis[spike_mask], y_demo_raw[spike_mask], color="red", zorder=5, label="detected spikes")
axes[0].set_xlabel("Raman shift (cm-1)"); axes[0].set_ylabel("Intensity (a.u.)")
axes[0].set_title("Step 1: cosmic-spike removal"); axes[0].legend(fontsize=8)

axes[1].plot(axis, y_desp, color="tab:orange", label="despiked (with baseline)")
axes[1].plot(axis, baseline_als, color="black", linestyle="--", label="ALS baseline (lam=1e6, p=0.01)")
axes[1].plot(axis, y_clean_als, color="tab:blue", label="baseline-removed")
axes[1].set_xlabel("Raman shift (cm-1)"); axes[1].set_ylabel("Intensity (a.u.)")
axes[1].set_title("Step 2: ALS baseline removal"); axes[1].legend(fontsize=8)
plt.tight_layout(); plt.show()

### λ 與 p 的效果：太小的 λ 會把真實波峰吃掉

下面固定 p=0.01，只改變 λ，觀察基線如何從「幾乎貼著原始光譜」（過度配適，overfitting）
逐漸變成「合理的平滑背景」。過度配適的基線會把 1003 cm⁻¹ 附近阿斯巴甜的真實波峰也當成背景扣掉，
導致扣除後波峰幾乎消失。

In [ ]:
lam_values = [1e2, 1e4, 1e6, 1e8]
idx_1003 = int(np.argmin(np.abs(axis - 1003)))
peak_height_raw = y_desp[idx_1003] - np.median(y_desp)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
retained_pct = []
for lam in lam_values:
    bl = als_baseline(y_desp, lam=lam, p=0.01, niter=10)
    corrected = y_desp - bl
    axes[0].plot(axis, bl, label=f"lam={lam:.0e}")
    axes[1].plot(axis, corrected, label=f"lam={lam:.0e}")
    peak_height = corrected[idx_1003] - corrected.min()
    retained_pct.append(100 * peak_height / (y_clean_als[idx_1003] - y_clean_als.min()))

axes[0].plot(axis, y_desp, color="grey", linewidth=1, label="raw (despiked)")
axes[0].set_title("ALS baseline for different lambda (p=0.01)")
axes[0].set_xlabel("Raman shift (cm-1)"); axes[0].legend(fontsize=7)
axes[1].axvline(1003, color="grey", linestyle=":", linewidth=1)
axes[1].set_title("Baseline-corrected spectra")
axes[1].set_xlabel("Raman shift (cm-1)"); axes[1].legend(fontsize=7)
plt.tight_layout(); plt.show()

for lam, pct in zip(lam_values, retained_pct):
    print(f"lam={lam:.0e}: 1003 cm-1 波峰保留約 {pct:.1f}% （相對 lam=1e6 的基準）")

可以看到 **λ 太小（如 1e2）時，基線幾乎貼著原始曲線走**，扣除後 1003 cm⁻¹ 的波峰高度大幅下降——
基線把真實訊號也「學」進去了，這就是過度配適 (overfitting)。λ 太大則基線會變得幾乎是一條直線，
可能來不及跟上真正的螢光背景曲率。實務上通常從 λ ≈ 10⁵–10⁷（依資料點數量與間距而定）開始嘗試，
再依「基線是否還貼著波峰」來調整。

### 向量正規化 (Vector Normalization)

基線移除後，光譜之間仍可能有整體強度差異（樣品量、對焦、積分時間不同）。向量正規化把每條光譜除以自己的
長度（$\lVert y \rVert$），讓比較聚焦在「形狀」而非「絕對強度」——這是進入相似度比對前的最後一步。

In [ ]:
def vector_normalize(y):
    n = np.linalg.norm(y)
    return y / n if n > 0 else y


y_norm = vector_normalize(y_clean_als)
print(f"基線移除後向量長度 = {np.linalg.norm(y_clean_als):.4f}")
print(f"正規化後向量長度   = {np.linalg.norm(y_norm):.4f}  (應為 1.0)")

## 3. NIR 前處理：SNV / MSC / Savitzky–Golay 平滑與導數

NIR 光譜的典型問題不是螢光基線，而是**粒徑造成的散射 (scatter)**：同一個成分，因為粉末顆粒大小或
裝填鬆緊不同，光路長度改變，讓光譜的**截距**與**斜率**跟著變。

- **SNV (Standard Normal Variate)**：用每條光譜自己的平均值與標準差做標準化，$y' = (y-\bar y)/s_y$。
- **MSC (Multiplicative Scatter Correction)**：把每條光譜對一條參考光譜（通常取平均光譜）做線性迴歸
  $y = a + b\,y_{ref}$，再用 $(y-a)/b$ 校正——概念與 SNV 類似，差別在用「參考光譜」而非「自身統計量」。

In [ ]:
NIR_WL = np.arange(900.0, 1700.0 + 1e-9, 2.0)
n_nir = len(NIR_WL)


def gauss_nir(x, c, w, h):
    return h * np.exp(-((x - c) ** 2) / (2.0 * w ** 2))


nir_shape = (gauss_nir(NIR_WL, 1200, 40, 0.5) + gauss_nir(NIR_WL, 1450, 35, 0.8)
             + gauss_nir(NIR_WL, 1650, 45, 0.35) + 0.1)
scales, offsets = [0.65, 1.0, 1.35], [0.05, 0.20, 0.42]
nir_raw_sets = [s * nir_shape + o for s, o in zip(scales, offsets)]


def snv(y):
    y = np.asarray(y, dtype=float)
    return (y - y.mean()) / y.std(ddof=0)


def msc(spectra, reference=None):
    """spectra: list/array of shape (n_spectra, n_points)."""
    spectra = np.asarray(spectra, dtype=float)
    ref = spectra.mean(axis=0) if reference is None else reference
    corrected = np.empty_like(spectra)
    for i, y in enumerate(spectra):
        b, a = np.polyfit(ref, y, 1)  # y = a + b*ref  (polyfit returns [b, a])
        corrected[i] = (y - a) / b
    return corrected


nir_snv_sets = [snv(y) for y in nir_raw_sets]
nir_msc_sets = list(msc(nir_raw_sets))

fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
for y, s in zip(nir_raw_sets, scales):
    axes[0].plot(NIR_WL, y, label=f"scale={s}")
for y, s in zip(nir_snv_sets, scales):
    axes[1].plot(NIR_WL, y, label=f"scale={s}")
for y, s in zip(nir_msc_sets, scales):
    axes[2].plot(NIR_WL, y, label=f"scale={s}")
for ax, title in zip(axes, ["Raw (scatter)", "After SNV", "After MSC"]):
    ax.set_xlabel("Wavelength (nm)"); ax.set_title(title); ax.legend(fontsize=8)
axes[0].set_ylabel("Intensity (a.u.)")
plt.tight_layout(); plt.show()

print("Raw std of 3 spectra:", [round(float(np.std(y)), 3) for y in nir_raw_sets])
print("SNV std of 3 spectra:", [round(float(np.std(y)), 3) for y in nir_snv_sets])

### Savitzky–Golay 平滑與 1st/2nd 導數

SG 濾波在一個滑動視窗內對資料做多項式擬合，同時可以直接取「擬合多項式的導數」當作平滑過的導數光譜。
兩個超參數：
- **視窗寬度 (window_length)**：越寬越平滑，但也可能把相鄰的窄波峰糊在一起（解析度下降）。
- **多項式階數 (polyorder)**：階數太低會抹平波峰形狀，太高則失去平滑效果（幾乎跟原始資料一樣）。

導數的好處是能把「重疊的寬波峰」拆解成清楚的極值，代價是**逐點雜訊被放大**——因為導數本質上是相鄰點的差分。

In [ ]:
nir2_shape = gauss_nir(NIR_WL, 1450, 25, 0.6) + gauss_nir(NIR_WL, 1500, 22, 0.45) + 0.08
rng_d = np.random.default_rng(3)
nir2_raw = nir2_shape + rng_d.normal(0, 0.004, n_nir)

smoothed = savgol_filter(nir2_raw, window_length=15, polyorder=2)
d1 = savgol_filter(nir2_raw, window_length=15, polyorder=2, deriv=1, delta=(NIR_WL[1] - NIR_WL[0]))
d2 = savgol_filter(nir2_raw, window_length=15, polyorder=2, deriv=2, delta=(NIR_WL[1] - NIR_WL[0]))

fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
axes[0].plot(NIR_WL, nir2_raw, color="grey", alpha=.6, label="raw")
axes[0].plot(NIR_WL, smoothed, color="tab:blue", label="SG smoothed")
axes[0].set_title("SG smoothing (window=15, poly=2)"); axes[0].legend(fontsize=8)
axes[1].plot(NIR_WL, d1, color="tab:green")
axes[1].set_title("1st derivative")
axes[2].plot(NIR_WL, d2, color="tab:red")
axes[2].axvline(1450, color="grey", linestyle=":"); axes[2].axvline(1500, color="grey", linestyle=":")
axes[2].set_title("2nd derivative (peaks -> negative minima)")
for ax in axes:
    ax.set_xlabel("Wavelength (nm)")
axes[0].set_ylabel("Intensity (a.u.)")
plt.tight_layout(); plt.show()

# window/polyorder trade-off + noise amplification
noisy = nir2_shape + rng_d.normal(0, 0.012, n_nir)
for win in (7, 15, 31):
    d2_win = savgol_filter(noisy, window_length=win, polyorder=2, deriv=2)
    print(f"window={win:>2d}: 2nd-derivative point-to-point std = {np.std(np.diff(d2_win)):.5f}")
print(f"(比較：原始雜訊光譜 point-to-point std = {np.std(np.diff(noisy)):.5f})")

## 4. 相似度指標 (Similarity Metrics)

| 指標 | 公式 | 對「強度大小」敏感嗎 |
|---|---|---|
| Pearson r | $\dfrac{\sum (x_i-\bar x)(y_i-\bar y)}{\sqrt{\sum(x_i-\bar x)^2}\sqrt{\sum(y_i-\bar y)^2}}$ | 不敏感（對平移、縮放都不敏感） |
| cosine | $\dfrac{x\cdot y}{\lVert x\rVert \lVert y\rVert}$ | 對縮放不敏感，但對平移敏感 |
| SAM (度) | $\arccos(\text{cosine}) \times \dfrac{180}{\pi}$ | 同 cosine，只是換算成角度 |
| Euclidean | $\lVert x-y \rVert$ | **非常敏感**——比對前務必先正規化 |

下面用一組刻意設計的示意向量，重現「原始 Euclidean 距離判斷錯誤、正規化後才判斷正確」的排名反轉 (ranking flip)。

In [ ]:
def pearson_r(a, b):
    return float(np.corrcoef(a, b)[0, 1])


def cosine_sim(a, b):
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))


def sam_deg(a, b):
    return float(np.degrees(np.arccos(np.clip(cosine_sim(a, b), -1.0, 1.0))))


def euclid(a, b):
    return float(np.linalg.norm(a - b))


vecU = np.array([0.10, 0.35, 0.90, 0.40, 0.15])   # 待測光譜(示意，5個資料點)
vecC1 = np.array([0.20, 0.55, 1.30, 0.60, 0.25])  # 正確化合物：形狀相同、強度較大
vecC2 = np.array([0.12, 0.32, 0.95, 0.38, 0.22])  # 錯誤化合物：形狀不同、但絕對數值接近 U

print(f"U vs C1:  Pearson r={pearson_r(vecU, vecC1):.4f}  cosine={cosine_sim(vecU, vecC1):.4f}  "
      f"SAM={sam_deg(vecU, vecC1):.2f}deg")
print(f"U vs C2:  Pearson r={pearson_r(vecU, vecC2):.4f}  cosine={cosine_sim(vecU, vecC2):.4f}  "
      f"SAM={sam_deg(vecU, vecC2):.2f}deg")

d_raw_C1, d_raw_C2 = euclid(vecU, vecC1), euclid(vecU, vecC2)
d_norm_C1 = euclid(vector_normalize(vecU), vector_normalize(vecC1))
d_norm_C2 = euclid(vector_normalize(vecU), vector_normalize(vecC2))

print(f"\n原始 Euclidean:    d(U,C1)={d_raw_C1:.4f}  d(U,C2)={d_raw_C2:.4f}  "
      f"-> 判斷最像的是 {'C1(正確)' if d_raw_C1 < d_raw_C2 else 'C2(錯誤!)'}")
print(f"正規化後 Euclidean: d(U,C1)={d_norm_C1:.4f}  d(U,C2)={d_norm_C2:.4f}  "
      f"-> 判斷最像的是 {'C1(正確)' if d_norm_C1 < d_norm_C2 else 'C2(錯誤!)'}")

Pearson r 與 cosine/SAM 從一開始就正確判斷 C1 較像（因為它們對「形狀」比較敏感）；
但**原始 Euclidean 距離卻判斷 C2 較像**，因為 C2 的絕對數值剛好離 U 比較近，即使形狀其實跟 U 不同。
正規化之後，Euclidean 才「回頭」給出正確答案——這正是「比對前一定要先正規化」的原因。

## 5. 建立光譜庫、鑑別未知樣品 (Spectral Library & Unknown Search)

流程：對光譜庫的每一條原始光譜、以及每一個待測未知樣品，套用**完全相同**的前處理
（去尖峰 → ALS 基線移除 → 正規化），再用相似度指標搜尋光譜庫，回報 **top-3**。

> **強調**：光譜庫與待測光譜若前處理不同（例如庫做了正規化、查詢忘記做），算出的分數會反映「前處理差異」而不是化學組成差異——結論就不可靠。

In [ ]:
def clean_spectrum(y, lam=1e6, p=0.01):
    y_desp, _ = despike(y, window=7, z_thresh=6.0)
    bl = als_baseline(y_desp, lam=lam, p=p, niter=10)
    return y_desp - bl


# 對光譜庫每個化合物：清洗每個重複，取平均當作模板 (template)
compounds = sorted(meta.loc[meta["role"] == "library", "class"].unique())
templates = {}
for compound in compounds:
    idx = meta.index[(meta["role"] == "library") & (meta["class"] == compound)]
    cleaned = [vector_normalize(clean_spectrum(X[i])) for i in idx]
    templates[compound] = np.mean(cleaned, axis=0)

plot_spectra(axis, np.array(list(templates.values())), list(templates.keys()),
             title="Library templates (after despike + ALS + normalize)",
             xlabel="Raman shift (cm-1)")
plt.show()

In [ ]:
def search_library(y_raw, templates, metric="pearson"):
    y_clean = vector_normalize(clean_spectrum(y_raw))
    scores = {}
    for name, tpl in templates.items():
        if metric == "pearson":
            scores[name] = pearson_r(y_clean, tpl)
        elif metric == "cosine":
            scores[name] = cosine_sim(y_clean, tpl)
        elif metric == "sam":
            scores[name] = -sam_deg(y_clean, tpl)  # 角度越小越像 -> 取負號讓「越大越像」
        elif metric == "euclid":
            scores[name] = -euclid(y_clean, tpl)   # 距離越小越像 -> 取負號
        else:
            raise ValueError(metric)
    ranked = sorted(scores.items(), key=lambda kv: kv[1], reverse=True)
    return ranked[:3]


for uid in ["unk_1", "unk_2", "unk_3"]:
    y_u = X[meta.index[meta["sample_id"] == uid][0]]
    top3_pearson = search_library(y_u, templates, "pearson")
    print(f"{uid}: top-3 (Pearson r) = {top3_pearson}")

`unk_1`、`unk_2` 應該都能清楚辨識出第一名（分數明顯領先第二名）。`unk_3` 則刻意設計成
top-1、top-2 分數接近的模糊案例——遇到這種情況，不能只看數字，還要回頭用**視覺疊圖**確認，
甚至考慮「這可能不是單一化合物，而是混合物」（下一節 NNLS 的動機）。

In [ ]:
uid = "unk_3"
y_u = X[meta.index[meta["sample_id"] == uid][0]]
y_u_clean = vector_normalize(clean_spectrum(y_u))
top3_u3 = search_library(y_u, templates, "pearson")
cand_names = [c for c, _ in top3_u3[:2]]

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(axis, y_u_clean, color="black", linewidth=2, label=f"{uid} (query)")
for name in cand_names:
    ax.plot(axis, templates[name], linewidth=1.3, alpha=0.8, label=f"template: {name}")
ax.set_xlabel("Raman shift (cm-1)"); ax.set_ylabel("Normalized intensity")
ax.set_title(f"{uid}: overlay with top-2 candidates ({top3_u3[0][1]:.3f} vs {top3_u3[1][1]:.3f})")
ax.legend(fontsize=9)
plt.show()

## 6. 混合物定量預覽：NNLS (Non-Negative Least Squares)

單一比對的分數通常只反映「最像的一個化合物」，容易忽略含量較低的次要成分。NNLS 把混合物光譜
表示成 $y \approx \sum_i c_i \cdot \text{ref}_i + \text{offset}$，用非負限制（成分含量不能是負的）
求解出每個成分的係數 $c_i$，可以再換算成比例。

**注意**：NNLS 拿來定量，前處理只需要**去尖峰＋基線移除**，**不要做向量正規化**——因為正規化會破壞
「係數 = 比例」這個線性關係，正規化只適用於「找形狀最像的化合物」這種定性比對。

In [ ]:
y_mix_raw = X[meta.index[meta["sample_id"] == "mix_demo"][0]]
true_sucralose_pct = meta.loc[meta["sample_id"] == "mix_demo", "sucralose_pct"].iloc[0]
true_malto_pct = meta.loc[meta["sample_id"] == "mix_demo", "maltodextrin_pct"].iloc[0]

y_mix_clean = clean_spectrum(y_mix_raw, lam=1e6, p=0.01)  # 注意：不正規化

A = np.column_stack([templates["sucralose"], templates["maltodextrin"], np.ones(len(axis))])
coef, _ = nnls(A, y_mix_clean)
pred = A @ coef
ss_res = float(np.sum((y_mix_clean - pred) ** 2))
ss_tot = float(np.sum((y_mix_clean - y_mix_clean.mean()) ** 2))
r2 = 1 - ss_res / ss_tot
proportions = coef[:2] / coef[:2].sum()

print(f"真實生成比例（僅此示範用，正式練習不會事先告訴你）：sucralose={true_sucralose_pct:.2f}, "
      f"maltodextrin={true_malto_pct:.2f}")
print(f"NNLS 係數：sucralose_coef={coef[0]:.4f}, maltodextrin_coef={coef[1]:.4f}, offset={coef[2]:.4f}")
print(f"換算比例（僅兩成分間相對比例）：sucralose={proportions[0]:.2f}, maltodextrin={proportions[1]:.2f}")
print(f"R^2 = {r2:.4f}")

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(axis, y_mix_clean, color="black", label="mixture (baseline-removed)")
ax.plot(axis, pred, color="tab:orange", linestyle="--", label="NNLS fit")
ax.set_xlabel("Raman shift (cm-1)"); ax.set_ylabel("Intensity (a.u.)")
ax.set_title(f"NNLS mixture fit (R^2={r2:.3f})")
ax.legend(fontsize=9)
plt.show()

注意：NNLS 用的模板（`templates["sucralose"]` 等）本身已經是**正規化過的形狀模板**，
所以這裡估計出來的係數只能拿來算「兩個成分之間的相對比例」，不能直接當作絕對濃度——
若要回報絕對濃度，需要用**已知濃度的標準品**重新建立未正規化的定量校正曲線（這是 W16 要做的事）。

## 7. 練習題 (Exercises)

請在每題下方的空白程式碼區塊中作答（可以用文字 print 說明，也可以搭配計算/繪圖）。

### 問題 1：尖峰移除視窗大小

對 `y_demo_raw`（第 2 節的示意光譜）分別用 `window=5` 與 `window=25` 執行 `despike()`，
比較兩者偵測到的尖峰數量與位置。哪一個視窗大小可能會誤判，把 1003 cm⁻¹ 附近的真實波峰也當成尖峰？

In [ ]:
# TODO: 請在這裡作答

### 問題 2：ALS 的 λ 與波峰保留率

延續第 2 節「λ 與 p 的效果」，改成固定 λ=1e6，改變 p ∈ {0.001, 0.01, 0.1}，
比較不同 p 值對基線形狀與 1003 cm⁻¹ 波峰保留率的影響，並說明 p 代表的意義。

In [ ]:
# TODO: 請在這裡作答

### 問題 3：unk_3 是單一化合物還是混合物？

對 `unk_3` 分別用 Pearson r、cosine、SAM、Euclidean（正規化後）四種指標算出 top-3，
比較四種指標給出的排名是否一致。根據分數的接近程度，你認為 `unk_3` 比較可能是單一化合物、
還是兩種化合物的混合物？請說明理由。

In [ ]:
# TODO: 請在這裡作答

### 問題 4：對 unk_3 做 NNLS

參考第 6 節的做法，選用問題 3 中找出的 top-2 候選化合物模板，對 `unk_3` 做 NNLS 擬合，
回報兩個成分的係數、換算比例，以及 R²。這個結果支持你在問題 3 的判斷嗎？

In [ ]:
# TODO: 請在這裡作答

### 問題 5：`mix_exercise` 的真實比例是多少？

資料庫中的 `mix_exercise`（`role == "mixture"`）刻意**沒有**在 CSV 裡標示真實的
`sucralose_pct` / `maltodextrin_pct`。請對它套用第 6 節同樣的 NNLS 流程，
估計出 sucralose 與 maltodextrin 的比例與 R²。

In [ ]:
# TODO: 請在這裡作答

## 8. 匯出結果 (Export)

把光譜庫搜尋結果與 NNLS 擬合結果存成 CSV。若在 Colab 環境執行，會另外觸發瀏覽器下載；
在本機執行則只會存檔，不會嘗試下載。

In [ ]:
export_rows = []
for uid in ["unk_1", "unk_2", "unk_3"]:
    y_u = X[meta.index[meta["sample_id"] == uid][0]]
    for rank, (name, score) in enumerate(search_library(y_u, templates, "pearson"), start=1):
        export_rows.append({"sample_id": uid, "rank": rank, "candidate": name, "pearson_r": score})

results_df = pd.DataFrame(export_rows)
output_path = "nb09_library_search_results.csv"
results_df.to_csv(output_path, index=False)
print(f"已儲存：{output_path}")
results_df

In [ ]:
try:
    from google.colab import files
    files.download(output_path)
except ImportError:
    print("非 Colab 環境，略過自動下載（檔案已存在本機/雲端硬碟工作目錄）。")